# SupeRR Selector Season 2 — Identifying Reliable Players Under Pressure
## Stage 0A: Raw CSV File Upload

In [1]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/SupeRR"  # update to your folder path

Mounted at /content/drive


In [4]:
import pandas as pd
d = pd.read_csv(f"{DATA_DIR}/ball_by_ball.csv", low_memory=False)
assert len(d) == 200999 and d["match_id"].nunique() == 867, (
    f"Truncated upload detected! Got {len(d)} rows / {d['match_id'].nunique()} matches instead of 200,999 / 867."
)

✅ Loaded 'ball_by_ball': 200,999 rows x 33 columns
✅ Loaded 'competition_team_player_mapping': 6,413 rows x 6 columns
✅ Loaded 'competitions': 141 rows x 4 columns
✅ Loaded 'matches': 883 rows x 22 columns
✅ Loaded 'playing_eleven': 28,381 rows x 9 columns
✅ Loaded 'teams': 353 rows x 3 columns


## Stage 0B: Load Raw DataFrames into Memory

In [ ]:
# Stage 0B: Load competition CSV files from Google Drive into a dictionary of DataFrames
# Inputs: DATA_DIR (str)
# Outputs: dfs (dict of raw DataFrames)

import os
import pandas as pd

expected_files = [
    "ball_by_ball.csv",
    "competition_team_player_mapping.csv",
    "competitions.csv",
    "matches.csv",
    "playing_eleven.csv",
    "teams.csv",
]

dfs = {}

for filename in sorted(expected_files):
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Missing file in Drive folder: {filepath}")
    name = filename.replace(".csv", "")
    dfs[name] = pd.read_csv(filepath, low_memory=False)
    print(
        f"✅ Loaded '{name}': {dfs[name].shape[0]:,} rows x"
        f" {dfs[name].shape[1]} columns"
    )

## Stage 0C: Table Normalization & Exploratory Data Audit

In [12]:
# Stage 0C: Normalize table keys and audit results, innings, extras, dismissals, and player IDs
# Inputs: dfs (dict)
# Outputs: tables (dict), b, m, c, p (raw base DataFrames)

import re
import pandas as pd

# Normalize dictionary keys, e.g. "ball_by_ball (1)" -> "ball_by_ball".
tables = {}
for name, df in dfs.items():
    clean_name = re.sub(r"\s*\(\d+\)$", "", name).strip().lower()
    if clean_name in tables:
        raise ValueError(f"Multiple files resolve to {clean_name!r}")
    tables[clean_name] = df.copy()

required = [
    "ball_by_ball",
    "competition_team_player_mapping",
    "competitions",
    "matches",
    "playing_eleven",
    "teams",
]
missing = set(required) - set(tables)
if missing:
    raise ValueError(f"Missing tables: {sorted(missing)}")

b = tables["ball_by_ball"]
m = tables["matches"]
c = tables["competitions"]
p = tables["competition_team_player_mapping"]


def show(title, data, limit=40):
    print(f"\n{'=' * 75}\n{title}\n{'=' * 75}")
    if isinstance(data, pd.Series):
        print(data.head(limit).to_string())
    else:
        print(data.head(limit).to_string(index=False))


# 1. Result labels: identify abandoned/no-result statuses without guessing.
show(
    "1. MATCH RESULT VALUES",
    m["result"].fillna("<missing>").value_counts(),
    limit=100,
)

# 2. Innings labels: inspect potential super-over encodings.
show(
    "2. INNINGS VALUES",
    b["innings"].value_counts(dropna=False),
)

show(
    "3. OVER RANGES BY INNINGS",
    b.groupby("innings", dropna=False).agg(
        rows=("match_id", "size"),
        matches=("match_id", "nunique"),
        min_over=("overid", "min"),
        max_over=("overid", "max"),
    ).reset_index(),
)

# 3. Inspect all extras labels and representative scoring combinations.
show(
    "4. EXTRAS TYPE VALUES",
    b["extras_type"].fillna("<missing>").value_counts(),
    limit=100,
)

score_cols = [
    "match_id", "innings", "overid", "over_ball_num",
    "striker_name", "runs", "is_four", "is_six",
    "extras_type", "extras1_runs", "extras_2_runs",
]

extra_mask = (
    b["extras_type"].notna()
    | pd.to_numeric(b["extras1_runs"], errors="coerce").fillna(0).ne(0)
    | pd.to_numeric(b["extras_2_runs"], errors="coerce").fillna(0).ne(0)
)

extra_examples = (
    b.loc[extra_mask, score_cols]
    .groupby("extras_type", dropna=False, sort=False)
    .head(5)
)
show("5. EXAMPLES FOR EACH EXTRAS TYPE", extra_examples, limit=100)

# Cases with runs beyond extras are especially useful for interpretation.
runs = pd.to_numeric(b["runs"], errors="coerce")
extras = (
    pd.to_numeric(b["extras1_runs"], errors="coerce").fillna(0)
    + pd.to_numeric(b["extras_2_runs"], errors="coerce").fillna(0)
)
show(
    "6. EXTRA DELIVERIES WITH RUNS GREATER THAN EXTRAS",
    b.loc[extra_mask & runs.gt(extras), score_cols],
    limit=15,
)

show(
    "7. DISMISSAL TYPES",
    b.groupby(
        ["dismissal_type_id", "dismissal_type"], dropna=False
    ).size().rename("rows").reset_index(),
    limit=100,
)

show("8. WICKET FLAG VALUES", b["is_wicket"].value_counts(dropna=False))

show(
    "9. COMPETITIONS AND SEASONS",
    c[["comp_id", "comp_name", "match_format", "season"]],
    limit=160,
)

# 4. Validate expected dimension-table uniqueness.
key_checks = []
for table, keys in [
    ("matches", ["match_id"]),
    ("competitions", ["comp_id"]),
    ("teams", ["comp_id", "team_id"]),
    ("competition_team_player_mapping", ["comp_id", "team_id", "player_id"]),
    ("playing_eleven", ["match_id", "team_id", "player_id"]),
]:
    df = tables[table]
    key_checks.append({
        "table": table,
        "rows": len(df),
        "duplicate_key_rows": int(df.duplicated(keys, keep=False).sum()),
        "rows_with_missing_key": int(df[keys].isna().any(axis=1).sum()),
    })

show("10. JOIN-KEY CHECKS", pd.DataFrame(key_checks))

# Do not delete repeated ball labels automatically:
# inspect them separately from exact duplicate records.
print("\nExact duplicate delivery rows:", int(b.duplicated().sum()))
print(
    "Rows sharing match/innings/over/ball labels:",
    int(
        b.duplicated(
            ["match_id", "innings", "overid", "over_ball_num"],
            keep=False,
        ).sum()
    ),
)

# 5. Player identity checks.
players = p[["player_id", "player_name"]].drop_duplicates().copy()
players["normalized_name"] = (
    players["player_name"].astype("string")
    .str.normalize("NFKC")
    .str.casefold()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

name_id_counts = (
    players.groupby("normalized_name")["player_id"].nunique()
)
ambiguous_names = name_id_counts[name_id_counts.gt(1)].index

show(
    "11. SAME NORMALIZED NAME WITH MULTIPLE PLAYER IDs",
    players.loc[players["normalized_name"].isin(ambiguous_names)]
    .sort_values(["normalized_name", "player_id"]),
    limit=60,
)

id_name_counts = players.groupby("player_id")["normalized_name"].nunique()
variant_ids = id_name_counts[id_name_counts.gt(1)].index

show(
    "12. SAME PLAYER ID WITH MULTIPLE NAME VARIANTS",
    players.loc[players["player_id"].isin(variant_ids)]
    .sort_values(["player_id", "normalized_name"]),
    limit=60,
)

# Familiar players: inspect identities and competition coverage.
known = p[
    p["player_name"].astype("string").str.contains(
        r"kohli|rohit sharma|jos buttler",
        case=False,
        na=False,
    )
]
show(
    "13. SPOT-CHECK PLAYER IDs AND COMPETITION COVERAGE",
    known[["player_id", "player_name", "comp_id", "comp_name"]]
    .drop_duplicates()
    .sort_values(["player_name", "comp_id"]),
    limit=100,
)


1. MATCH RESULT VALUES
result
No Result                                       21
Match Abandoned                                 19
India Won by 7 wickets                           8
Delhi Capitals Won by 6 wickets                  7
Kolkata Knight Riders Won by 8 wickets           6
Punjab Kings Won by 5 wickets                    5
India Won by 6 Wickets                           5
Gujarat Titans Won by 7 wickets                  5
West Indies Won by 5 wickets                     4
Australia Won by 3 wickets                       4
Rajasthan Royals Won by 6 wickets                4
New Zealand Won by 8 wickets                     4
Punjab Kings Won by 7 wickets                    4
Bangladesh Won by 8 wickets                      4
Bangladesh Won by 4 wickets                      4
Mumbai Indians Won by 6 wickets                  4
Pakistan Won by 7 wickets                        4
Royal Challengers Bengaluru Won by 6 wickets     4
England Won by 4 wickets                         3


## Stage 0D: Schema Verification for Core Tables

In [13]:
# Stage 0D: Verify exact column names for ball_by_ball and matches
# Inputs: dfs (dict)
# Outputs: Printed column lists for schema confirmation

for name in ["ball_by_ball", "matches"]:
    key = next(
        k for k in dfs
        if k == name or k.startswith(f"{name} (")
    )
    print(f"{name}: {list(dfs[key].columns)}")

ball_by_ball: ['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time']
matches: ['match_id', 'comp_id', 'match_number', 'match_date', 'stage', 'venue_id', 'venue_country', 'home_team_id', 'home_team', 'away_team_id', 'away_team', 'home_team_captain_id', 'away_team_captain_id', 'toss_win_team_id', 'toss_opted', 'man_of_the_match', 'first_inn_revised_overs', 'second_inn_revised_overs', 'revised_target', 'knock_out', 'win_team_id', 'result']


## Stage 1: Match-Level Flags & Exclusion Rules

In [14]:
# Stage 1: Merge competition metadata and flag match-level exclusions and tiers
# Inputs: m, c, b (raw DataFrames)
# Outputs: match_flags (match-level DataFrame with keep_match and exclusion_reason)

import numpy as np
import pandas as pd

# Work on a copy so raw tables stay intact
match_flags = m.copy().merge(
    c[["comp_id", "comp_name", "match_format", "season"]],
    on="comp_id",
    how="left",
    validate="many_to_one",
)

match_flags["match_date"] = pd.to_datetime(match_flags["match_date"], errors="coerce")

# Check empty-value encoding before deciding what counts as rain-revised
rain_cols = ["first_inn_revised_overs", "second_inn_revised_overs", "revised_target"]
print("Rain revision encoding:")
for col in rain_cols:
    s = match_flags[col]
    n_null = int(s.isna().sum())
    s_str = s.dropna().astype(str).str.strip()
    n_blank = int((s_str == "").sum())
    n_zero = int(pd.to_numeric(s, errors="coerce").eq(0).sum())
    unique_non_null = sorted(s.dropna().unique().tolist(), key=str)[:20]
    print(f"  {col} ({s.dtype}): nulls={n_null}, blanks={n_blank}, zeros={n_zero}, sample={unique_non_null}")

# Separate valid IPL and T20I matches from emerging-team T20 tournaments
excluded_comps = {
    "Asia Cup Rising Stars 2025",
    "Mens T20 Emerging Teams Asia Cup 2024",
}
comp_name_clean = match_flags["comp_name"].fillna("").str.strip()
is_ipl = comp_name_clean.str.match(r"^IPL \d{4}$")
is_t20i = match_flags["match_format"].fillna("").str.strip().eq("T20I")
is_valid_comp = (is_ipl | is_t20i) & ~comp_name_clean.isin(excluded_comps)

match_flags["comp_type"] = np.select(
    [is_ipl, is_t20i & ~comp_name_clean.isin(excluded_comps)],
    ["IPL", "T20I"],
    default="",
)

# Tag match tier for analysis splits without dropping associate games
full_members = {
    "India", "Australia", "England", "South Africa", "New Zealand", "Pakistan",
    "West Indies", "Sri Lanka", "Bangladesh", "Afghanistan", "Zimbabwe", "Ireland",
}
home_clean = match_flags["home_team"].fillna("").str.strip()
away_clean = match_flags["away_team"].fillna("").str.strip()
both_full = home_clean.isin(full_members) & away_clean.isin(full_members)

match_flags["tier"] = np.select(
    [
        match_flags["comp_type"].eq("IPL"),
        match_flags["comp_type"].eq("T20I") & both_full,
        match_flags["comp_type"].eq("T20I") & ~both_full,
    ],
    ["IPL", "full_member", "associate"],
    default="",
)

# Case-insensitive check prevents capitalization variants from slipping through
result_clean = match_flags["result"].fillna("").str.strip().str.casefold()
is_no_result = result_clean.isin({"no result", "match abandoned"})

# Treat NaN, blank text, and 0 as unrevised because actual revisions are positive
rain_non_empty = pd.DataFrame(index=match_flags.index)
for col in rain_cols:
    s = match_flags[col]
    has_text = s.notna() & (s.astype(str).str.strip() != "")
    is_zero = pd.to_numeric(s, errors="coerce").eq(0)
    rain_non_empty[col] = has_text & ~is_zero
is_rain_revised = rain_non_empty.any(axis=1)

# Both regular innings are needed to evaluate chases and verify final margins
inn_present = (
    b.loc[b["innings"].isin([1, 2])]
    .groupby("match_id")["innings"]
    .nunique()
)
has_both_innings = match_flags["match_id"].map(inn_present).fillna(0).eq(2)

# First matching reason wins; empty string means kept
match_flags["exclusion_reason"] = np.select(
    [
        ~is_valid_comp,
        is_no_result,
        is_rain_revised,
        ~has_both_innings,
    ],
    [
        "not IPL/T20I",
        "no result / abandoned",
        "rain revised",
        "missing innings",
    ],
    default="",
)
match_flags["keep_match"] = match_flags["exclusion_reason"].eq("")

total_matches = len(match_flags)
kept_matches = int(match_flags["keep_match"].sum())
dropped_matches = total_matches - kept_matches

print(f"\nMatches total: {total_matches} | kept: {kept_matches} | dropped: {dropped_matches}")

print("\nDropped per reason:")
print(
    match_flags.loc[~match_flags["keep_match"], "exclusion_reason"]
    .value_counts()
    .to_string()
)

print("\nKept matches by comp_type and tier:")
print(
    match_flags.loc[match_flags["keep_match"]]
    .groupby(["comp_type", "tier"])
    .size()
    .rename("matches")
    .reset_index()
    .to_string(index=False)
)

# Confirm no full member landed in associate due to a naming mismatch
assoc_rows = match_flags.loc[match_flags["keep_match"] & match_flags["tier"].eq("associate")]
assoc_teams = sorted(
    (set(assoc_rows["home_team"].dropna()) | set(assoc_rows["away_team"].dropna())) - full_members
)
print("\nNon-full-member teams in kept associate T20Is:", assoc_teams)

# Flag kept matches without 'won' in result (ties, super overs, or missing labels)
kept_non_win = match_flags.loc[
    match_flags["keep_match"] & ~match_flags["result"].fillna("").str.contains("won", case=False)
]
print(f"\nKept matches without 'won' in result: {len(kept_non_win)}")
if len(kept_non_win) > 0:
    print(kept_non_win[["match_id", "comp_name", "match_date", "result", "win_team_id"]].to_string(index=False))

print("\nMissing match_date values:", int(match_flags["match_date"].isna().sum()))

Rain revision encoding:
  first_inn_revised_overs (float64): nulls=858, blanks=0, zeros=0, sample=[102.0, 108.0, 114.0, 30.0, 42.0, 48.0, 54.0, 60.0, 66.0, 72.0, 78.0, 84.0, 90.0]
  second_inn_revised_overs (float64): nulls=858, blanks=0, zeros=0, sample=[102.0, 114.0, 30.0, 42.0, 48.0, 54.0, 60.0, 66.0, 72.0, 78.0, 84.0, 90.0]
  revised_target (float64): nulls=860, blanks=0, zeros=0, sample=[103.0, 104.0, 115.0, 116.0, 125.0, 126.0, 130.0, 136.0, 143.0, 147.0, 151.0, 152.0, 168.0, 171.0, 203.0, 213.0, 33.0, 56.0, 69.0, 78.0]

Matches total: 883 | kept: 789 | dropped: 94

Dropped per reason:
exclusion_reason
no result / abandoned    40
not IPL/T20I             30
rain revised             24

Kept matches by comp_type and tier:
comp_type        tier  matches
      IPL         IPL      357
     T20I   associate       50
     T20I full_member      382

Non-full-member teams in kept associate T20Is: ['Canada', 'Hong Kong', 'Kuwait', 'Nepal', 'Netherlands', 'Oman', 'Scotland', 'Singapore', 

## Stage 2: Delivery-Level Base Table & Chronological Ordering

In [15]:
# Stage 2: Filter to regular innings (1-2) of kept matches and verify ball-order monotonicity
# Inputs: b, match_flags
# Outputs: deliveries_flagged, d (sorted delivery-level DataFrame)

import numpy as np
import pandas as pd

# Flag deliveries on a copy of b so raw ball_by_ball stays untouched
deliveries_flagged = b.copy().merge(
    match_flags[["match_id", "exclusion_reason", "keep_match"]],
    on="match_id",
    how="left",
    validate="many_to_one",
)

# Match-level exclusions take priority; remaining innings 3-4 rows are super overs
missing_match = deliveries_flagged["keep_match"].isna()
match_excluded = deliveries_flagged["exclusion_reason"].fillna("").ne("")
is_super_over = ~deliveries_flagged["innings"].isin([1, 2])

deliveries_flagged["delivery_exclusion_reason"] = np.select(
    [missing_match, match_excluded, is_super_over],
    ["missing match", deliveries_flagged["exclusion_reason"], "super over"],
    default="",
)
deliveries_flagged["keep_delivery"] = deliveries_flagged["delivery_exclusion_reason"].eq("")

rows_before = len(deliveries_flagged)

# Build the filtered base table only from the flag column
d = (
    deliveries_flagged.loc[deliveries_flagged["keep_delivery"]]
    .drop(columns=["exclusion_reason", "keep_match", "delivery_exclusion_reason", "keep_delivery"])
    .copy()
)

match_info_cols = [
    "match_id", "comp_id", "comp_type", "tier",
    "season", "match_date", "venue_id", "knock_out", "win_team_id", "result",
]
d = d.merge(
    match_flags[match_info_cols],
    on="match_id",
    how="left",
    validate="many_to_one",
)

rows_after = len(d)

print(f"Deliveries before: {rows_before:,} | after: {rows_after:,} | removed: {rows_before - rows_after:,}")
print("\nDeliveries removed per reason:")
print(
    deliveries_flagged.loc[~deliveries_flagged["keep_delivery"], "delivery_exclusion_reason"]
    .value_counts()
    .to_string()
)

# Check whether innings_ball_num is strictly increasing within each (match_id, innings)
null_ball_num = int(d["innings_ball_num"].isna().sum())
dup_mask = d.duplicated(["match_id", "innings", "innings_ball_num"], keep=False)
dup_ball_num = int(dup_mask.sum())

prev_ball_num = d.groupby(["match_id", "innings"], sort=False)["innings_ball_num"].shift(1)
raw_decrease_mask = d["innings_ball_num"].le(prev_ball_num)
raw_decreases = int(raw_decrease_mask.sum())

print(
    f"\ninnings_ball_num check: nulls={null_ball_num}, "
    f"duplicate_key_rows={dup_ball_num}, non_increasing_steps_in_raw_order={raw_decreases}"
)

if null_ball_num == 0 and dup_ball_num == 0 and raw_decreases == 0:
    # Sorting locks in chronological order for cumulative state calculations in Stage 4
    d = d.sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)
    print("Strictly increasing within every (match_id, innings): YES. Sorted d.")
else:
    print("Strictly increasing check FAILED. Inspect problem rows below before sorting:")
    bad_groups = d.loc[
        d["innings_ball_num"].isna() | dup_mask | raw_decrease_mask,
        ["match_id", "innings"],
    ].drop_duplicates()
    print(f"Affected (match_id, innings) groups: {len(bad_groups)}")
    sample_keys = bad_groups.head(5)
    sample_rows = d.merge(sample_keys, on=["match_id", "innings"], how="inner")
    # Show only rows near the issue in each affected innings
    for (mid, inn), grp in sample_rows.groupby(["match_id", "innings"], sort=False):
        g = grp.copy()
        g["prev_ball_num"] = g["innings_ball_num"].shift(1)
        g["is_issue"] = (
            g["innings_ball_num"].isna()
            | g["innings_ball_num"].duplicated(keep=False)
            | g["innings_ball_num"].le(g["prev_ball_num"])
        )
        issue_pos = np.where(g["is_issue"])[0]
        window = sorted({i for p in issue_pos[:3] for i in range(max(0, p - 1), min(len(g), p + 2))})
        cols = ["match_id", "innings", "overid", "over_ball_num", "innings_ball_num", "extras_type", "striker_name"]
        print(f"\nSample issue in match_id={mid}, innings={inn}:")
        print(g.iloc[window][cols].to_string(index=False))

print(f"\nKept unique matches in d: {d['match_id'].nunique()}")

Deliveries before: 200,999 | after: 188,330 | removed: 12,669

Deliveries removed per reason:
delivery_exclusion_reason
not IPL/T20I             6985
rain revised             3905
no result / abandoned    1753
super over                 26

innings_ball_num check: nulls=0, duplicate_key_rows=0, non_increasing_steps_in_raw_order=0
Strictly increasing within every (match_id, innings): YES. Sorted d.

Kept unique matches in d: 789


## Stage 3: Delivery-Level Flags & Scoring Hypotheses

In [16]:
# Stage 3: Construct legal-ball, extras, run, and dismissal flags on delivery table d
# Inputs: d
# Outputs: d with delivery flags and team_runs Hypotheses A and B

import numpy as np
import pandas as pd

# Confirm run and extra columns have no unexpected nulls before arithmetic
num_cols = ["runs", "extras1_runs", "extras_2_runs"]
print("Null counts in scoring columns:", d[num_cols].isna().sum().to_dict())

runs = d["runs"].fillna(0)
ext1 = d["extras1_runs"].fillna(0)
ext2 = d["extras_2_runs"].fillna(0)

# Inspect how runs and both extra columns behave under every extras_type
extras_summary = (
    d.assign(extras_type_clean=d["extras_type"].fillna("<missing>"))
    .groupby("extras_type_clean", dropna=False)
    .agg(
        rows=("match_id", "size"),
        runs_gt0=("runs", lambda s: int(s.fillna(0).gt(0).sum())),
        ext1_vals=("extras1_runs", lambda s: sorted(s.dropna().unique().tolist())[:10]),
        ext2_gt0=("extras_2_runs", lambda s: int(s.fillna(0).gt(0).sum())),
        ext2_vals=("extras_2_runs", lambda s: sorted(s.dropna().unique().tolist())[:10]),
    )
    .reset_index()
)
print("\nScoring breakdown by extras_type:")
print(extras_summary.to_string(index=False))

# Check whether "other" behaves like a no-ball (re-bowled delivery) or a legal ball
other_mask = d["extras_type"].eq("other")
print(f"\nRows with extras_type == 'other': {int(other_mask.sum())}")
print("Sample 'other' deliveries:")
sample_cols = [
    "match_id", "innings", "overid", "over_ball_num",
    "runs", "extras1_runs", "extras_2_runs", "is_wicket", "dismissal_type", "delivery_style",
]
print(d.loc[other_mask, sample_cols].head(10).to_string(index=False))

# For completed (non-final) overs containing "other", a true no-ball leaves 6 remaining legal balls
max_over_per_inn = d.groupby(["match_id", "innings"])["overid"].transform("max")
is_non_final_over = d["overid"].lt(max_over_per_inn)
overs_with_other = (
    d.loc[other_mask & is_non_final_over, ["match_id", "innings", "overid"]]
    .drop_duplicates()
)
if len(overs_with_other) > 0:
    other_overs_df = d.merge(overs_with_other, on=["match_id", "innings", "overid"], how="inner")
    other_overs_df["legal_if_other_is_nb"] = ~other_overs_df["extras_type"].isin(["wide", "Overstep", "other"])
    other_overs_df["legal_if_other_is_legal"] = ~other_overs_df["extras_type"].isin(["wide", "Overstep"])
    over_check = other_overs_df.groupby(["match_id", "innings", "overid"]).agg(
        if_nb=("legal_if_other_is_nb", "sum"),
        if_legal=("legal_if_other_is_legal", "sum"),
    )
    print("\nNon-final overs containing 'other' — legal ball count under both interpretations:")
    print("  Treating 'other' as no-ball (expect 6):")
    print(over_check["if_nb"].value_counts().sort_index().to_string())
    print("  Treating 'other' as legal ball (expect 7 if it was re-bowled):")
    print(over_check["if_legal"].value_counts().sort_index().to_string())

# Check rows where extras_type is missing but extras1_runs or extras_2_runs is non-zero
unlabeled_extras = d["extras_type"].isna() & (ext1.gt(0) | ext2.gt(0))
print(f"\nRows with missing extras_type but non-zero extras runs: {int(unlabeled_extras.sum())}")
if unlabeled_extras.sum() > 0:
    print(d.loc[unlabeled_extras, sample_cols].head(10).to_string(index=False))

# Build delivery flags (hypotheses to be validated against match results in Stage 5)
d["is_wide"] = d["extras_type"].eq("wide")
d["is_noball"] = d["extras_type"].isin(["Overstep", "other"])

# Wides and no-balls must be re-bowled, so they do not count toward the 6-ball over
d["is_legal"] = ~(d["is_wide"] | d["is_noball"])

# Batters can score off no-balls (they count as balls faced), but not off wides
d["batter_faced"] = ~d["is_wide"]
d["batter_runs"] = runs

# Keep both team-run hypotheses so Stage 5 can test whether extras_2_runs belongs in the total
d["team_runs"] = runs + ext1 + ext2
d["team_runs_hyp_b"] = runs + ext1

# Bowlers concede batter runs plus wide and no-ball penalties; byes, legbyes, and team penalties are excluded
# For no-balls, extras_2_runs are byes/legbyes off the no-ball so they are excluded from bowler_runs
wide_runs = np.where(d["is_wide"], ext1 + ext2, 0)
noball_runs = np.where(d["is_noball"], ext1, 0)
d["bowler_runs"] = np.where(
    d["extras_type"].eq("Penalty"),
    0,
    runs + wide_runs + noball_runs,
)

is_wicket_bool = d["is_wicket"].fillna(False).astype(bool)
bowler_dismissals = {"Bowled", "Caught", "LBW", "Stump", "Hit Wicket"}
d["is_bowler_wicket"] = is_wicket_bool & d["dismissal_type"].isin(bowler_dismissals)

# Retire Out is excluded per Stage 3 hypothesis (will also check team wicket logic in Stage 5)
d["is_team_wicket"] = is_wicket_bool & ~d["dismissal_type"].eq("Retire Out")

# Run-outs can dismiss the non-striker, so use dismissed_player_id rather than striker_id
d["dismissed_id"] = d["dismissed_player_id"]

# Print value counts for each flag
print("\nBoolean flag value counts:")
flag_cols = [
    "is_wide", "is_noball", "is_legal",
    "batter_faced", "is_bowler_wicket", "is_team_wicket",
]
for col in flag_cols:
    vc = d[col].value_counts(dropna=False).to_dict()
    print(f"  {col}: {vc}")

print("\nRun column value counts:")
for col in ["batter_runs", "team_runs", "team_runs_hyp_b", "bowler_runs"]:
    vc = d[col].value_counts(dropna=False).sort_index().to_dict()
    print(f"  {col}: {vc}")

print(
    f"\ndismissed_id non-null count: {int(d['dismissed_id'].notna().sum())} "
    f"(is_wicket True count: {int(is_wicket_bool.sum())})"
)

# Check distribution of legal balls per over (non-final overs should almost always have 6)
legal_per_over = (
    d.groupby(["match_id", "innings", "overid"])["is_legal"]
    .sum()
)
print("\nLegal balls per over (all overs):")
print(legal_per_over.value_counts().sort_index().to_string())

non_final_legal_per_over = (
    d.loc[is_non_final_over]
    .groupby(["match_id", "innings", "overid"])["is_legal"]
    .sum()
)
print("\nLegal balls per over (excluding final over of each innings):")
print(non_final_legal_per_over.value_counts().sort_index().to_string())

Null counts in scoring columns: {'runs': 0, 'extras1_runs': 0, 'extras_2_runs': 0}

Scoring breakdown by extras_type:
extras_type_clean   rows  runs_gt0       ext1_vals  ext2_gt0 ext2_vals
        <missing> 177210    113627          [0, 1]         1    [0, 1]
         Overstep    545       319             [1]        21 [0, 1, 4]
             byes    485         0    [1, 2, 4, 5]         0       [0]
          legbyes   2571         0 [1, 2, 3, 4, 5]         0       [0]
            other    254       132             [1]        13 [0, 1, 4]
             wide   7265         0 [1, 2, 3, 4, 5]         0       [0]

Rows with extras_type == 'other': 254
Sample 'other' deliveries:
 match_id  innings  overid  over_ball_num  runs  extras1_runs  extras_2_runs  is_wicket dismissal_type delivery_style
    13513        1      10              1     0             1              0      False            NaN      Full Toss
    13515        1      16              6     0             1              0      F

## Stage 4: DLS/Tie Refinement, Dismissal Gating & Pre-Delivery Match State

In [17]:
# Stage 4: Apply DLS/tie match filters, gate dismissals, and compute pre-delivery match state once
# Inputs: match_flags (Stage 1), d (Stage 3, with knock_out and validated team_runs)
# Outputs: match_flags_final (883-row audit table), d_state (state-enriched delivery table)

import numpy as np
import pandas as pd

# Work on a copy of Stage 1 flags so Stage 1 outputs and Appendix Stage 5 stay intact
match_flags_final = match_flags.copy()

# Compute regular-innings totals from Stage 3 validated team_runs (Hypothesis A)
inn_totals = (
    d.groupby(["match_id", "innings"])["team_runs"]
    .sum()
    .unstack("innings")
)
match_flags_final["inn1_total"] = match_flags_final["match_id"].map(inn_totals[1])
match_flags_final["inn2_total"] = match_flags_final["match_id"].map(inn_totals[2])

# Evaluate DLS and tie rules only on matches that survived Stage 1 (first matching reason wins)
is_still_kept = match_flags_final["exclusion_reason"].eq("")
result_clean = match_flags_final["result"].fillna("").str.strip().str.casefold()

# Catch DLS/D/L matches whose revised-target columns were left empty in matches
is_dls_text = match_flags_final["result"].fillna("").str.contains(r"dls|d/l", case=False, regex=True)

# Exclude equal-score ties and rain-par ties so the binary chase outcome has no ambiguity
scores_equal = (
    match_flags_final["inn1_total"].notna()
    & match_flags_final["inn2_total"].notna()
    & match_flags_final["inn1_total"].eq(match_flags_final["inn2_total"])
)
is_tied = scores_equal | result_clean.eq("match tied")

match_flags_final["exclusion_reason"] = np.select(
    [~is_still_kept, is_dls_text, is_tied],
    [match_flags_final["exclusion_reason"], "DLS / D/L", "tied match"],
    default="",
)
match_flags_final["keep_match"] = match_flags_final["exclusion_reason"].eq("")

# Score-derived label avoids result-text typos, super-over strings, and missing win_team_id
match_flags_final["chasing_team_won"] = np.where(
    match_flags_final["keep_match"],
    (match_flags_final["inn2_total"] > match_flags_final["inn1_total"]).astype(int),
    np.nan,
)

# Filter deliveries strictly via the keep_match flag
d_state = d.merge(
    match_flags_final[["match_id", "keep_match", "chasing_team_won"]],
    on="match_id",
    how="left",
    validate="many_to_one",
)
d_state = (
    d_state.loc[d_state["keep_match"]]
    .drop(columns=["keep_match"])
    .sort_values(["match_id", "innings", "innings_ball_num"])
    .reset_index(drop=True)
)
d_state["chasing_team_won"] = d_state["chasing_team_won"].astype(int)

# Gate dismissed_id on is_wicket so overturned/non-wicket rows with stray IDs are not counted
is_wicket_bool = d_state["is_wicket"].fillna(False).astype(bool)
phantom_dismissals = d_state.loc[~is_wicket_bool & d_state["dismissed_id"].notna()]
print(f"Rows with dismissed_id populated but is_wicket=False (clearing): {len(phantom_dismissals)}")
d_state["dismissed_id"] = d_state["dismissed_player_id"].where(is_wicket_bool, np.nan)

# Subtract current ball from cumulative sums so each row reflects match state BEFORE the delivery
grp_cols = ["match_id", "innings"]
team_wkt_int = d_state["is_team_wicket"].astype(int)
legal_int = d_state["is_legal"].astype(int)

d_state["cum_runs"] = d_state.groupby(grp_cols)["team_runs"].cumsum() - d_state["team_runs"]
d_state["cum_wickets"] = team_wkt_int.groupby([d_state["match_id"], d_state["innings"]]).cumsum() - team_wkt_int
d_state["legal_balls_bowled"] = legal_int.groupby([d_state["match_id"], d_state["innings"]]).cumsum() - legal_int
d_state["balls_remaining"] = 120 - d_state["legal_balls_bowled"]

# Leave overs outside 1-20 as NaN rather than forcing an invalid phase label
d_state["phase"] = pd.cut(
    d_state["overid"],
    bins=[0, 6, 15, 20],
    labels=["powerplay", "middle", "death"],
).astype("object")

# Compute innings-2 chase state; leave innings 1 and zero-denominator rates as NaN
inn1_totals_s4 = d_state.loc[d_state["innings"].eq(1)].groupby("match_id")["team_runs"].sum()
is_inn2 = d_state["innings"].eq(2)

d_state["target"] = np.where(is_inn2, d_state["match_id"].map(inn1_totals_s4) + 1, np.nan)
d_state["runs_needed"] = np.where(is_inn2, d_state["target"] - d_state["cum_runs"], np.nan)
d_state["current_rr"] = np.where(
    is_inn2 & d_state["legal_balls_bowled"].gt(0),
    d_state["cum_runs"] * 6.0 / d_state["legal_balls_bowled"],
    np.nan,
)

# Cap required_rr at 36.0 (six sixes an over) so final-ball outliers do not distort logistic models
raw_rrr = np.where(
    is_inn2 & d_state["balls_remaining"].gt(0),
    d_state["runs_needed"] * 6.0 / d_state["balls_remaining"],
    np.nan,
)
n_rrr_capped = int((raw_rrr > 36.0).sum())
d_state["required_rr"] = np.minimum(raw_rrr, 36.0)

# Verification checks
print(
    f"\nMatches total: {len(match_flags_final)} | kept: {int(match_flags_final['keep_match'].sum())} "
    f"| dropped: {int((~match_flags_final['keep_match']).sum())}"
)
print("\nDropped per reason:")
print(match_flags_final.loc[~match_flags_final["keep_match"], "exclusion_reason"].value_counts().to_string())

newly_excluded = match_flags_final.loc[match_flags_final["exclusion_reason"].isin(["DLS / D/L", "tied match"])]
print(f"\nMatches excluded by DLS / D/L or tied match ({len(newly_excluded)}):")
print(
    newly_excluded[["match_id", "comp_type", "match_date", "inn1_total", "inn2_total", "exclusion_reason", "result"]]
    .to_string(index=False)
)

print("\nModel label balance across kept matches (chasing_team_won):")
print(
    match_flags_final.loc[match_flags_final["keep_match"], "chasing_team_won"]
    .astype(int)
    .value_counts()
    .sort_index()
    .to_string()
)

print(
    f"\nDismissal gate check — dismissed_id non-null: {int(d_state['dismissed_id'].notna().sum())} "
    f"| is_wicket True: {int(is_wicket_bool.sum())}"
)
print(f"required_rr check — rows capped at 36.0: {n_rrr_capped} | max required_rr: {d_state['required_rr'].max():.2f}")
print("knock_out value counts in d_state:", d_state["knock_out"].value_counts(dropna=False).to_dict())

print("\nPhase counts:")
print(d_state["phase"].value_counts(dropna=False).to_string())

print("\nInnings-2 state sanity bounds:")
state_cols = [
    "cum_runs", "cum_wickets", "legal_balls_bowled", "balls_remaining",
    "target", "runs_needed", "current_rr", "required_rr",
]
print(d_state.loc[is_inn2, state_cols].agg(["min", "max", "count"]).T.to_string())

Rows with dismissed_id populated but is_wicket=False (clearing): 10

Matches total: 883 | kept: 777 | dropped: 106

Dropped per reason:
exclusion_reason
no result / abandoned    40
not IPL/T20I             30
rain revised             24
tied match                8
DLS / D/L                 4

Matches excluded by DLS / D/L or tied match (12):
 match_id comp_type match_date  inn1_total  inn2_total exclusion_reason                                                            result
    13613      T20I 2022-02-13       164.0       164.0       tied match                             Australia Won the one over Eliminator
    18532      T20I 2022-11-22       160.0        75.0       tied match                                                        Match Tied
    20182       IPL 2023-04-01       191.0       146.0        DLS / D/L                            Punjab Kings Won by 7 Runs(D/L Method)
    21176      T20I 2023-04-02       196.0       196.0       tied match                                 

## Stage 6: Master Table & Exclusion Log Export

In [18]:
# Stage 6: Export final exclusion_log.csv and master_deliveries.parquet from Stage 4 outputs
# Inputs: match_flags_final, d_state
# Outputs: exclusion_log.csv, master_deliveries (DataFrame + parquet/csv)

import os
from google.colab import files

# Full 883-match audit trail (kept matches have empty exclusion_reason)
exclusion_log = match_flags_final[["match_id", "exclusion_reason"]].copy()
exclusion_log_path = "exclusion_log.csv"
exclusion_log.to_csv(exclusion_log_path, index=False)

# Drop the losing run-total hypothesis so downstream scripts cannot pick it by mistake
master_deliveries = d_state.drop(columns=["team_runs_hyp_b"], errors="ignore").copy()

assert "knock_out" in master_deliveries.columns, "knock_out missing from master_deliveries"

master_path = "master_deliveries.parquet"
try:
    master_deliveries.to_parquet(master_path, index=False)
except Exception as err:
    print(f"Parquet export failed ({err}); falling back to CSV.")
    master_path = "master_deliveries.csv"
    master_deliveries.to_csv(master_path, index=False)

print(f"Saved {exclusion_log_path}: shape={exclusion_log.shape}, size={os.path.getsize(exclusion_log_path) / 1024:.1f} KB")
print(f"Saved {master_path}: shape={master_deliveries.shape}, size={os.path.getsize(master_path) / (1024 * 1024):.2f} MB")
print("\nFinal master_deliveries columns:")
print(list(master_deliveries.columns))

files.download(exclusion_log_path)
files.download(master_path)

Saved exclusion_log.csv: shape=(883, 2), size=7.6 KB
Saved master_deliveries.parquet: shape=(185567, 62), size=4.14 MB

Final master_deliveries columns:
['match_id', 'innings', 'overid', 'over_ball_num', 'innings_ball_num', 'batting_team_id', 'bowling_team_id', 'striker_id', 'striker_name', 'non_striker_id', 'non_striker_name', 'bowler_id', 'bowler_name', 'batting_type', 'bowling_type', 'batting_order', 'runs', 'is_four', 'is_six', 'is_wicket', 'dismissal_type_id', 'dismissal_type', 'dismissed_player_id', 'extras_type', 'extras1_runs', 'extras_2_runs', 'pace_or_spin', 'bowl_speed', 'stroke', 'delivery_style', 'over_or_around', 'on_or_off', 'delivery_ist_time', 'comp_id', 'comp_type', 'tier', 'season', 'match_date', 'venue_id', 'knock_out', 'win_team_id', 'result', 'is_wide', 'is_noball', 'is_legal', 'batter_faced', 'batter_runs', 'team_runs', 'bowler_runs', 'is_bowler_wicket', 'is_team_wicket', 'dismissed_id', 'chasing_team_won', 'cum_runs', 'cum_wickets', 'legal_balls_bowled', 'balls_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Stage 7: Out-of-Fold Win Probability Modeling (GroupKFold Logistic Regression)

In [19]:
# Stage 7: Train 5-fold match-grouped logistic regression models for Innings 1 and Innings 2 win probability
# Inputs: master_deliveries
# Outputs: wp (DataFrame with out-of-fold p_bat_win_before), log-loss, and calibration tables

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Load from disk only if the Colab session restarted
if "master_deliveries" not in globals():
    master_deliveries = pd.read_parquet("master_deliveries.parquet")

# Work on a copy so master_deliveries stays untouched
wp = master_deliveries.copy()

# Compute regular-innings totals directly from team_runs so labels never depend on result text
inn_totals = (
    wp.groupby(["match_id", "innings"])["team_runs"]
    .sum()
    .unstack("innings")
)
wp["inn1_total"] = wp["match_id"].map(inn_totals[1])
wp["inn2_total"] = wp["match_id"].map(inn_totals[2])

# Filter defensively in case an older master_deliveries file is loaded
is_tier_ok = wp["tier"].isin(["IPL", "full_member"])
is_dls = wp["result"].fillna("").str.contains(r"dls|d/l", case=False, regex=True)
is_tie = wp["inn1_total"].eq(wp["inn2_total"]) | wp["result"].fillna("").str.strip().str.casefold().eq("match tied")

keep_mask = is_tier_ok & ~is_dls & ~is_tie
print(
    f"Filter summary — matches: {wp['match_id'].nunique()} -> {wp.loc[keep_mask, 'match_id'].nunique()} "
    f"| deliveries: {len(wp):,} -> {int(keep_mask.sum()):,}"
)
print(
    f"Dropped deliveries — non-IPL/associate: {int((~is_tier_ok).sum()):,}, "
    f"DLS: {int((is_tier_ok & is_dls).sum()):,}, "
    f"tied: {int((is_tier_ok & ~is_dls & is_tie).sum()):,}"
)

wp = wp.loc[keep_mask].sort_values(["match_id", "innings", "innings_ball_num"]).reset_index(drop=True)

# Match-level binary targets derived strictly from innings totals
wp["chase_won"] = (wp["inn2_total"] > wp["inn1_total"]).astype(int)
wp["bat_first_won"] = 1 - wp["chase_won"]

# Shared features
wp["wickets_in_hand"] = 10 - wp["cum_wickets"]
wp["is_IPL"] = wp["comp_type"].eq("IPL").astype(int)

# Recompute current_rr across both innings so innings 1 is 0 before the first legal ball
wp["wp_current_rr"] = np.where(
    wp["legal_balls_bowled"].gt(0),
    wp["cum_runs"] * 6.0 / wp["legal_balls_bowled"],
    0.0,
)

# Cap required_rr at 36 so extreme last-ball rates do not distort linear logits
wp["wp_required_rr"] = np.minimum(wp["required_rr"], 36.0)

# Interaction terms for chase and innings-1 state
wp["rrr_x_wkts"] = wp["wp_required_rr"] * wp["wickets_in_hand"]
wp["balls_x_wkts"] = wp["balls_remaining"] * wp["wickets_in_hand"]
wp["runs_x_balls"] = wp["cum_runs"] * wp["balls_remaining"]

inn1_features = [
    "cum_runs",
    "balls_remaining",
    "wickets_in_hand",
    "wp_current_rr",
    "runs_x_balls",
    "balls_x_wkts",
    "is_IPL",
]
inn2_features = [
    "runs_needed",
    "balls_remaining",
    "wickets_in_hand",
    "wp_required_rr",
    "rrr_x_wkts",
    "balls_x_wkts",
    "is_IPL",
]

def fit_oof_win_model(sub_df, feature_cols, target_col, n_splits=5):
    X = sub_df[feature_cols]
    y = sub_df[target_col].values
    groups = sub_df["match_id"].values

    nulls = X.isna().sum()
    if nulls.any():
        raise ValueError(f"Missing feature values found:\n{nulls[nulls > 0]}")

    # Group by match_id so no player or match is evaluated by a model trained on the same game
    gkf = GroupKFold(n_splits=n_splits)
    oof_pred = np.empty(len(sub_df), dtype=float)
    oof_base = np.empty(len(sub_df), dtype=float)
    fold_coefs = []

    for tr_idx, va_idx in gkf.split(X, y, groups=groups):
        # Standardize inside each fold so L2 regularization treats all feature scales evenly without leakage
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000)),
        ])
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        oof_pred[va_idx] = pipe.predict_proba(X.iloc[va_idx])[:, 1]
        oof_base[va_idx] = y[tr_idx].mean()
        fold_coefs.append(pipe.named_steps["clf"].coef_[0])

    model_ll = log_loss(y, oof_pred)
    base_ll = log_loss(y, oof_base)

    # 10 fixed probability bins to check calibration across the full 0-1 range
    bins = np.linspace(0.0, 1.0, 11)
    cal = (
        pd.DataFrame({"pred": oof_pred, "actual": y})
        .assign(bin=lambda df: pd.cut(df["pred"], bins=bins, include_lowest=True))
        .groupby("bin", observed=False)
        .agg(
            balls=("actual", "size"),
            mean_pred=("pred", "mean"),
            actual_rate=("actual", "mean"),
        )
        .reset_index()
    )
    cal["diff"] = cal["mean_pred"] - cal["actual_rate"]

    mean_coefs = pd.Series(np.mean(fold_coefs, axis=0), index=feature_cols)
    return oof_pred, model_ll, base_ll, cal, mean_coefs

inn1_mask = wp["innings"].eq(1)
inn2_mask = wp["innings"].eq(2)

oof_inn1, ll_inn1, base_ll_inn1, cal_inn1, coef_inn1 = fit_oof_win_model(
    wp.loc[inn1_mask], inn1_features, "bat_first_won"
)
oof_inn2, ll_inn2, base_ll_inn2, cal_inn2, coef_inn2 = fit_oof_win_model(
    wp.loc[inn2_mask], inn2_features, "chase_won"
)

# Each model predicts the batting side's win probability before that delivery
wp["p_bat_win_before"] = np.nan
wp.loc[inn1_mask, "p_bat_win_before"] = oof_inn1
wp.loc[inn2_mask, "p_bat_win_before"] = oof_inn2

print(f"\nMissing p_bat_win_before: {int(wp['p_bat_win_before'].isna().sum())}")

print("\nLog-loss vs constant out-of-fold base-rate model:")
ll_summary = pd.DataFrame([
    {
        "model": "Innings 1 (bat_first_won)",
        "balls": int(inn1_mask.sum()),
        "model_logloss": ll_inn1,
        "base_logloss": base_ll_inn1,
        "improvement": base_ll_inn1 - ll_inn1,
    },
    {
        "model": "Innings 2 (chase_won)",
        "balls": int(inn2_mask.sum()),
        "model_logloss": ll_inn2,
        "base_logloss": base_ll_inn2,
        "improvement": base_ll_inn2 - ll_inn2,
    },
])
print(ll_summary.round(4).to_string(index=False))

print("\nCalibration table — Innings 1 (bat_first_won):")
print(cal_inn1.round({"mean_pred": 4, "actual_rate": 4, "diff": 4}).to_string(index=False))

print("\nCalibration table — Innings 2 (chase_won):")
print(cal_inn2.round({"mean_pred": 4, "actual_rate": 4, "diff": 4}).to_string(index=False))

# First delivery of each innings checks pre-innings baseline calibration
ball1 = wp.groupby(["match_id", "innings"], sort=False).head(1)
ball1_summary = (
    ball1.assign(
        bat_won=np.where(ball1["innings"].eq(1), ball1["bat_first_won"], ball1["chase_won"])
    )
    .groupby("innings")
    .agg(
        matches=("match_id", "nunique"),
        mean_p_ball1=("p_bat_win_before", "mean"),
        min_p_ball1=("p_bat_win_before", "min"),
        max_p_ball1=("p_bat_win_before", "max"),
        actual_bat_win_rate=("bat_won", "mean"),
    )
    .reset_index()
)
print("\nMean p_bat_win_before at ball 1 of each innings:")
print(ball1_summary.round(4).to_string(index=False))

print("\nMean standardized coefficients (sanity check on feature signs):")
print("  Innings 1:", coef_inn1.round(3).to_dict())
print("  Innings 2:", coef_inn2.round(3).to_dict())

Filter summary — matches: 777 -> 727 | deliveries: 185,567 -> 173,920
Dropped deliveries — non-IPL/associate: 11,647, DLS: 0, tied: 0

Missing p_bat_win_before: 0

Log-loss vs constant out-of-fold base-rate model:
                    model  balls  model_logloss  base_logloss  improvement
Innings 1 (bat_first_won)  90490         0.6344        0.6997       0.0653
    Innings 2 (chase_won)  83430         0.3960        0.6934       0.2974

Calibration table — Innings 1 (bat_first_won):
          bin  balls  mean_pred  actual_rate    diff
(-0.001, 0.1]    615     0.0797       0.0504  0.0293
   (0.1, 0.2]   4473     0.1588       0.1786 -0.0198
   (0.2, 0.3]   8967     0.2531       0.2465  0.0066
   (0.3, 0.4]  13277     0.3530       0.3735 -0.0206
   (0.4, 0.5]  19342     0.4525       0.4521  0.0005
   (0.5, 0.6]  19462     0.5495       0.5616 -0.0120
   (0.6, 0.7]  12834     0.6454       0.6528 -0.0074
   (0.7, 0.8]   6829     0.7452       0.7035  0.0417
   (0.8, 0.9]   3747     0.8435     

## Stage 8: WPA, Empirically Smoothed Leverage (m=30) & Normalized WPA

In [20]:
# Stage 8: Compute delivery WPA, empirically smoothed leverage (m=30), high-pressure flags, and norm_wpa
# Inputs: wp (from Stage 7, includes knock_out)
# Outputs: df_wpa (with wpa, smoothed leverage, high_pressure flags, norm_wpa) and phase validation table

import numpy as np
import pandas as pd

# Work on a copy so Stage 7 output stays intact if Stage 8 is re-run
df_wpa = (
    wp.copy()
    .sort_values(["match_id", "innings", "innings_ball_num"])
    .reset_index(drop=True)
)

# Next ball's pre-ball probability gives post-ball state for every non-final delivery
next_p = df_wpa.groupby(["match_id", "innings"], sort=False)["p_bat_win_before"].shift(-1)

# Innings-1 batting team's win probability at the break is 1 minus the chasing team's ball-1 probability
inn2_ball1_p = (
    df_wpa.loc[df_wpa["innings"].eq(2)]
    .groupby("match_id", sort=False)["p_bat_win_before"]
    .first()
)
inn1_end_p = 1.0 - df_wpa["match_id"].map(inn2_ball1_p)
inn2_end_p = df_wpa["chase_won"].astype(float)

is_last_ball = next_p.isna()
df_wpa["wp_after"] = np.where(
    ~is_last_ball,
    next_p,
    np.where(df_wpa["innings"].eq(1), inn1_end_p, inn2_end_p),
)

# Zero-sum credit: a shift favoring the batting side hurts the bowling side equally
df_wpa["wpa"] = df_wpa["wp_after"] - df_wpa["p_bat_win_before"]
df_wpa["batter_wpa"] = df_wpa["wpa"]
df_wpa["bowler_wpa"] = -df_wpa["wpa"]
df_wpa["abs_wpa"] = df_wpa["wpa"].abs()

# Check 1: mean wpa overall and by innings (should sit near 0 for calibrated probabilities)
print(f"Missing wp_after: {int(df_wpa['wp_after'].isna().sum())}")
print(f"Overall mean wpa: {df_wpa['wpa'].mean():.6f}")
print("Mean wpa by innings:")
print(df_wpa.groupby("innings")["wpa"].mean().round(6).to_string())

# Check 2: telescoping identity per innings (sum of wpa must equal end_wp - start_wp)
inn_check = (
    df_wpa.groupby(["match_id", "innings"], sort=False)
    .agg(
        wpa_sum=("wpa", "sum"),
        start_wp=("p_bat_win_before", "first"),
        end_wp=("wp_after", "last"),
        chase_won=("chase_won", "first"),
        bat_first_won=("bat_first_won", "first"),
    )
    .reset_index()
)
inn_check["expected_end_minus_start"] = inn_check["end_wp"] - inn_check["start_wp"]
inn_check["telescoping_err"] = (inn_check["wpa_sum"] - inn_check["expected_end_minus_start"]).abs()

inn2_chk = inn_check.loc[inn_check["innings"].eq(2)]
inn2_outcome_err = (inn2_chk["wpa_sum"] - (inn2_chk["chase_won"] - inn2_chk["start_wp"])).abs().max()

match_pivot = inn_check.pivot(index="match_id", columns="innings", values=["wpa_sum", "start_wp", "bat_first_won"])
match_err = (
    (match_pivot[("wpa_sum", 1)] - match_pivot[("wpa_sum", 2)])
    - (match_pivot[("bat_first_won", 1)] - match_pivot[("start_wp", 1)])
).abs().max()

print("\nPer-innings telescoping check (max |sum(wpa) - (end_wp - start_wp)|):")
print(inn_check.groupby("innings")["telescoping_err"].max().to_string())
print(f"Innings 2 exact (outcome - start_wp) max abs error: {inn2_outcome_err:.2e}")
print(f"Full-match conservation max abs error (inn1_wpa - inn2_wpa vs outcome - inn1_start): {match_err:.2e}")

# Leverage bins: match state before the ball determines how much a single delivery can swing the game
df_wpa["wkt_bucket"] = pd.cut(
    df_wpa["cum_wickets"],
    bins=[-1, 2, 5, 10],
    labels=["0-2", "3-5", "6+"],
)
df_wpa["wp_decile"] = pd.qcut(
    df_wpa["p_bat_win_before"],
    q=10,
    labels=False,
    duplicates="drop",
)

fine_cols = ["innings", "overid", "wkt_bucket", "wp_decile"]
parent_cols = ["innings", "phase", "wkt_bucket", "wp_decile"]

# Empirical Bayes smoothing shrinks sparse fine bins (<10 balls) toward their coarser phase parent bin
m_prior = 30.0
fine_grp = df_wpa.groupby(fine_cols, observed=True)["abs_wpa"]
mu_fine = fine_grp.transform("mean")
n_fine = fine_grp.transform("size")
mu_parent = df_wpa.groupby(parent_cols, observed=True)["abs_wpa"].transform("mean")

df_wpa["smoothed_abs_wpa"] = (n_fine * mu_fine + m_prior * mu_parent) / (n_fine + m_prior)

# Normalize so leverage = 1.0 represents an average-stakes delivery across the dataset
df_wpa["leverage"] = df_wpa["smoothed_abs_wpa"] / df_wpa["smoothed_abs_wpa"].mean()

# Compute leverage-normalized WPA once so downstream stages do not recompute it (guard zero division)
df_wpa["norm_wpa"] = np.where(df_wpa["leverage"].gt(0), df_wpa["wpa"] / df_wpa["leverage"], 0.0)
df_wpa["batter_norm_wpa"] = df_wpa["norm_wpa"]
df_wpa["bowler_norm_wpa"] = -df_wpa["norm_wpa"]

fine_bin_stats = (
    df_wpa.groupby(fine_cols, observed=True)["abs_wpa"]
    .agg(bin_balls="size", bin_mean_abs_wpa="mean")
    .reset_index()
)
parent_bin_stats = (
    df_wpa.groupby(parent_cols, observed=True)["abs_wpa"]
    .agg(bin_balls="size", bin_mean_abs_wpa="mean")
    .reset_index()
)

print(
    f"\nLeverage smoothing diagnostics (m={int(m_prior)}) — "
    f"fine bins: {len(fine_bin_stats)} (median balls={fine_bin_stats['bin_balls'].median():.0f}, <10 balls={int(fine_bin_stats['bin_balls'].lt(10).sum())}) | "
    f"parent bins: {len(parent_bin_stats)} (median balls={parent_bin_stats['bin_balls'].median():.0f}, <10 balls={int(parent_bin_stats['bin_balls'].lt(10).sum())})"
)
print(
    f"Overall mean leverage: {df_wpa['leverage'].mean():.6f} "
    f"| min={df_wpa['leverage'].min():.4f}, max={df_wpa['leverage'].max():.4f}"
)
print("Mean leverage by innings:")
print(df_wpa.groupby("innings")["leverage"].mean().round(4).to_string())
print(
    f"norm_wpa check — nulls: {int(df_wpa['norm_wpa'].isna().sum())}, "
    f"mean: {df_wpa['norm_wpa'].mean():.6f}, std: {df_wpa['norm_wpa'].std():.6f}"
)

# Flag top leverage cutoffs within each innings so innings-1 pressure moments are not drowned out by chases
flag_specs = [
    (0.75, "high_pressure"),
    (0.80, "high_pressure_20"),
    (0.65, "high_pressure_35"),
]
flag_rows = []
for q, col_name in flag_specs:
    thresh = df_wpa.groupby("innings")["leverage"].transform(lambda s: s.quantile(q))
    df_wpa[col_name] = df_wpa["leverage"].ge(thresh)
    for inn, grp in df_wpa.groupby("innings"):
        flag_rows.append({
            "flag": col_name,
            "innings": inn,
            "leverage_cutoff": grp["leverage"].quantile(q),
            "flagged_pct": 100.0 * grp[col_name].mean(),
            "flagged_balls": int(grp[col_name].sum()),
        })

print("\nPressure flag cutoffs and actual flagged shares per innings:")
print(pd.DataFrame(flag_rows).round({"leverage_cutoff": 3, "flagged_pct": 2}).to_string(index=False))

# Validation table: compare high-pressure vs ordinary deliveries within each phase
is_dot = df_wpa["team_runs"].eq(0)
is_boundary = df_wpa["is_four"].fillna(False).astype(bool) | df_wpa["is_six"].fillna(False).astype(bool)

val_df = df_wpa.assign(
    phase_ord=pd.Categorical(df_wpa["phase"], categories=["powerplay", "middle", "death"], ordered=True),
    pressure_group=np.where(df_wpa["high_pressure"], "high_pressure", "ordinary"),
    is_dot=is_dot,
    is_boundary=is_boundary,
)

validation_table = (
    val_df.groupby(["phase_ord", "pressure_group"], observed=True)
    .agg(
        balls=("wpa", "size"),
        runs_per_ball=("team_runs", "mean"),
        dot_pct=("is_dot", lambda s: 100.0 * s.mean()),
        boundary_pct=("is_boundary", lambda s: 100.0 * s.mean()),
        wicket_pct=("is_team_wicket", lambda s: 100.0 * s.mean()),
        mean_abs_wpa=("abs_wpa", "mean"),
        mean_leverage=("leverage", "mean"),
    )
    .reset_index()
    .rename(columns={"phase_ord": "phase"})
)

print("\nValidation table — high-pressure vs ordinary within each phase:")
print(
    validation_table.round({
        "runs_per_ball": 3,
        "dot_pct": 2,
        "boundary_pct": 2,
        "wicket_pct": 2,
        "mean_abs_wpa": 4,
        "mean_leverage": 3,
    }).to_string(index=False)
)

Missing wp_after: 0
Overall mean wpa: -0.000078
Mean wpa by innings:
innings
1   -0.000089
2   -0.000066

Per-innings telescoping check (max |sum(wpa) - (end_wp - start_wp)|):
innings
1    0.000000e+00
2    1.110223e-16
Innings 2 exact (outcome - start_wp) max abs error: 1.11e-16
Full-match conservation max abs error (inn1_wpa - inn2_wpa vs outcome - inn1_start): 2.22e-16

Leverage smoothing diagnostics (m=30) — fine bins: 845 (median balls=113, <10 balls=130) | parent bins: 149 (median balls=485, <10 balls=6)
Overall mean leverage: 1.000000 | min=0.0778, max=11.6223
Mean leverage by innings:
innings
1    0.8523
2    1.1602
norm_wpa check — nulls: 0, mean: -0.000015, std: 0.027245

Pressure flag cutoffs and actual flagged shares per innings:
            flag  innings  leverage_cutoff  flagged_pct  flagged_balls
   high_pressure        1            1.045        25.43          23014
   high_pressure        2            1.546        25.09          20932
high_pressure_20        1          

## Stage 10: Player ID Deduplication Audit & Sensitivity Analysis

In [21]:
# Stage 10: Audit duplicate player names/IDs, merge verified split IDs, and run sensitivity and bootstrap checks
# Inputs: df_wpa, p
# Outputs: audit_df, id_merge_map, intermediate top-10 rankings, sensitivity metrics, and bootstrap CIs

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Work on a copy so Stage 8 output stays untouched
s10 = df_wpa.copy()

# ------------------------------------------------------------------
# 1. Audit and merge true duplicate player_ids (never merge different people)
# ------------------------------------------------------------------
players_p = p[["player_id", "player_name", "team_name", "comp_id"]].drop_duplicates().copy()
players_p["norm_name"] = (
    players_p["player_name"].astype("string")
    .str.normalize("NFKC")
    .str.casefold()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

dup_names = (
    players_p.groupby("norm_name")["player_id"]
    .nunique()
    .loc[lambda s: s.gt(1)]
    .index
)

bat_counts = s10.groupby("striker_id").size()
bowl_counts = s10.groupby("bowler_id").size()
bat_styles = s10.groupby("striker_id")["batting_type"].agg(lambda s: s.dropna().mode().iloc[0] if len(s.dropna()) else "")
bowl_styles = s10.groupby("bowler_id")["bowling_type"].agg(lambda s: s.dropna().mode().iloc[0] if len(s.dropna()) else "")

# Track matches per player_id so two players who appeared in the same match are never merged
matches_by_pid = {}
for col in ["striker_id", "non_striker_id", "bowler_id"]:
    for pid, grp in s10.groupby(col)["match_id"]:
        matches_by_pid.setdefault(pid, set()).update(grp.unique())

audit_rows = []
id_merge_map = {}

for norm_name, grp in players_p.loc[players_p["norm_name"].isin(dup_names)].groupby("norm_name"):
    pids = sorted(grp["player_id"].unique().tolist())
    pid_info = []
    for pid in pids:
        teams = sorted(grp.loc[grp["player_id"].eq(pid), "team_name"].dropna().unique().tolist())
        b_cnt = int(bat_counts.get(pid, 0))
        w_cnt = int(bowl_counts.get(pid, 0))
        pid_info.append({
            "norm_name": norm_name,
            "player_id": pid,
            "player_name": grp.loc[grp["player_id"].eq(pid), "player_name"].iloc[0],
            "teams_in_p": ", ".join(teams),
            "team_set": set(teams),
            "bat_balls_s8": b_cnt,
            "bowl_balls_s8": w_cnt,
            "total_balls_s8": b_cnt + w_cnt,
            "bat_style": bat_styles.get(pid, ""),
            "bowl_style": bowl_styles.get(pid, ""),
        })

    # Merge only when two IDs share at least one team and never appear in the same match
    pid_info_sorted = sorted(pid_info, key=lambda r: (-r["total_balls_s8"], r["player_id"]))
    primary = pid_info_sorted[0]
    for row in pid_info_sorted:
        action = "primary"
        if row["player_id"] != primary["player_id"]:
            shared_team = len(row["team_set"] & primary["team_set"]) > 0
            same_match = len(matches_by_pid.get(row["player_id"], set()) & matches_by_pid.get(primary["player_id"], set())) > 0
            if shared_team and not same_match:
                id_merge_map[row["player_id"]] = primary["player_id"]
                action = f"merged -> {primary['player_id']}"
            else:
                action = "kept separate (no shared team)" if not shared_team else "kept separate (same match)"
        audit_rows.append({k: v for k, v in row.items() if k != "team_set"} | {"action": action})

audit_df = pd.DataFrame(audit_rows)
print("Duplicate player-name audit:")
print(audit_df.drop(columns=["norm_name"]).to_string(index=False))
print("\nApplied player_id merge map:", id_merge_map)

if id_merge_map:
    for id_col in ["striker_id", "non_striker_id", "bowler_id", "dismissed_id"]:
        if id_col in s10.columns:
            s10[id_col] = s10[id_col].replace(id_merge_map)

# Build a unique player_id -> player_name map after any merges
id_to_name = (
    p[["player_id", "player_name"]]
    .dropna(subset=["player_id"])
    .assign(player_id=lambda df: df["player_id"].replace(id_merge_map))
    .drop_duplicates(subset=["player_id"])
    .set_index("player_id")["player_name"]
)

# Role-specific ball filters (same definitions as Stage 9)
bat_balls = s10.loc[s10["batter_faced"]].copy()

non_bowler_dismissals = {
    "RunOut",
    "Retire Out",
    "Obstructing the field",
    "Obstracting the field",
}
is_wkt = s10["is_wicket"].fillna(False).astype(bool)
bowl_balls = s10.loc[~(is_wkt & s10["dismissal_type"].isin(non_bowler_dismissals))].copy()


# ------------------------------------------------------------------
# 2. Final eligibility and Empirical Bayes scoring
# ------------------------------------------------------------------
def score_players(
    sub_df,
    id_col,
    fallback_name_col,
    wpa_col,
    pressure_col="high_pressure",
    min_press=150,
    min_total=300,
    min_matches=12,
):
    is_press = sub_df[pressure_col].astype(bool)
    tmp = pd.DataFrame({
        "player_id": sub_df[id_col].values,
        "match_id": sub_df["match_id"].values,
        "is_press": is_press.astype(int).values,
        "is_ord": (~is_press).astype(int).values,
        "wpa_press": np.where(is_press, sub_df[wpa_col].values, 0.0),
        "wpa_ord": np.where(~is_press, sub_df[wpa_col].values, 0.0),
    })

    agg = tmp.groupby("player_id", as_index=False).agg(
        matches=("match_id", "nunique"),
        n_total=("is_press", "size"),
        n_press=("is_press", "sum"),
        n_ord=("is_ord", "sum"),
        sum_wpa_press=("wpa_press", "sum"),
        sum_wpa_ord=("wpa_ord", "sum"),
    )
    agg["mean_wpa_pressure"] = np.where(agg["n_press"].gt(0), agg["sum_wpa_press"] / agg["n_press"], np.nan)
    agg["mean_wpa_ordinary"] = np.where(agg["n_ord"].gt(0), agg["sum_wpa_ord"] / agg["n_ord"], np.nan)

    fallback_names = (
        sub_df[[id_col, fallback_name_col]]
        .drop_duplicates(subset=[id_col])
        .set_index(id_col)[fallback_name_col]
    )
    agg["player_name"] = agg["player_id"].map(id_to_name).fillna(agg["player_id"].map(fallback_names))

    elig = agg.loc[
        agg["n_press"].ge(min_press)
        & agg["n_total"].ge(min_total)
        & agg["matches"].ge(min_matches)
    ].copy()

    # Shrink lower-volume players toward the eligible group mean using median pressure volume
    prior_mean = elig["mean_wpa_pressure"].mean()
    k = elig["n_press"].median()

    elig["shrunk_pressure_wpa"] = (
        elig["n_press"] * elig["mean_wpa_pressure"] + k * prior_mean
    ) / (elig["n_press"] + k)
    elig["pressure_lift"] = elig["shrunk_pressure_wpa"] - elig["mean_wpa_ordinary"]

    z_shrunk = (elig["shrunk_pressure_wpa"] - elig["shrunk_pressure_wpa"].mean()) / elig["shrunk_pressure_wpa"].std(ddof=0)
    z_lift = (elig["pressure_lift"] - elig["pressure_lift"].mean()) / elig["pressure_lift"].std(ddof=0)

    elig["final_score"] = 0.85 * z_shrunk + 0.15 * z_lift
    elig = elig.sort_values("final_score", ascending=False).reset_index(drop=True)
    elig.insert(0, "rank", np.arange(1, len(elig) + 1))

    meta = {"n_eligible": len(elig), "prior_mean": prior_mean, "k": k}
    return elig, meta


final_bat, bat_meta = score_players(bat_balls, "striker_id", "striker_name", "batter_wpa", "high_pressure")
final_bowl, bowl_meta = score_players(bowl_balls, "bowler_id", "bowler_name", "bowler_wpa", "high_pressure")

display_cols = [
    "rank", "player_id", "player_name", "matches", "n_press",
    "mean_wpa_ordinary", "mean_wpa_pressure", "shrunk_pressure_wpa", "final_score",
]

print(f"\nFinal Eligible Batters: {bat_meta['n_eligible']} | prior_mean={bat_meta['prior_mean']:.6f} | k={bat_meta['k']:.1f}")
print("Final Top 10 Batters:")
print(
    final_bat[display_cols].head(10).round({
        "mean_wpa_ordinary": 5, "mean_wpa_pressure": 5,
        "shrunk_pressure_wpa": 5, "final_score": 3,
    }).to_string(index=False)
)

print(f"\nFinal Eligible Bowlers: {bowl_meta['n_eligible']} | prior_mean={bowl_meta['prior_mean']:.6f} | k={bowl_meta['k']:.1f}")
print("Final Top 10 Bowlers:")
print(
    final_bowl[display_cols].head(10).round({
        "mean_wpa_ordinary": 5, "mean_wpa_pressure": 5,
        "shrunk_pressure_wpa": 5, "final_score": 3,
    }).to_string(index=False)
)


# ------------------------------------------------------------------
# 3a. Threshold sensitivity: 20% and 35% leverage vs main 25%
# ------------------------------------------------------------------
def compare_rankings(main_df, alt_df, label):
    top10_main = set(main_df.head(10)["player_id"])
    top10_alt = set(alt_df.head(10)["player_id"])
    overlap = len(top10_main & top10_alt)

    merged = main_df[["player_id", "final_score"]].merge(
        alt_df[["player_id", "final_score"]],
        on="player_id",
        suffixes=("_main", "_alt"),
    )
    rho, pval = spearmanr(merged["final_score_main"], merged["final_score_alt"])
    return {
        "comparison": label,
        "alt_eligible": len(alt_df),
        "common_eligible": len(merged),
        "top10_overlap": f"{overlap}/10",
        "spearman_rho": round(float(rho), 4),
    }

sens_rows = []
for role, df_role, id_col, name_col, wpa_col, main_df in [
    ("Batters", bat_balls, "striker_id", "striker_name", "batter_wpa", final_bat),
    ("Bowlers", bowl_balls, "bowler_id", "bowler_name", "bowler_wpa", final_bowl),
]:
    for col, tag in [("high_pressure_20", "Top 20% leverage"), ("high_pressure_35", "Top 35% leverage")]:
        alt_df, _ = score_players(df_role, id_col, name_col, wpa_col, pressure_col=col)
        sens_rows.append({"role": role, **compare_rankings(main_df, alt_df, tag)})

print("\nThreshold sensitivity vs Main (Top 25% leverage):")
print(pd.DataFrame(sens_rows).to_string(index=False))


# ------------------------------------------------------------------
# 3b. Simple non-ML composite index comparison
# ------------------------------------------------------------------
# Score margin (runs_needed) is only defined in Innings 2 because Innings 1 has no target
simple_press_mask = (
    s10["innings"].eq(2)
    & (
        (s10["required_rr"].ge(10.0) & s10["cum_wickets"].ge(3))
        | (s10["overid"].ge(16) & s10["runs_needed"].le(20))
    )
)
s10["simple_pressure"] = simple_press_mask
print(f"\nSimple non-ML pressure deliveries: {int(simple_press_mask.sum()):,} ({100.0 * simple_press_mask.mean():.2f}% of all balls)")

# Evaluate on the same eligible player pool (requiring >= 30 simple-pressure balls to avoid tiny-sample noise)
min_simple_balls = 30

bat_simple_src = s10.loc[s10["batter_faced"] & s10["simple_pressure"]]
bat_simple = (
    bat_simple_src.groupby("striker_id", as_index=False)
    .agg(simple_balls=("batter_runs", "size"), simple_runs=("batter_runs", "sum"))
    .rename(columns={"striker_id": "player_id"})
)
bat_simple = final_bat[["player_id", "player_name", "final_score"]].merge(bat_simple, on="player_id", how="inner")
bat_simple = bat_simple.loc[bat_simple["simple_balls"].ge(min_simple_balls)].copy()
bat_simple["simple_score"] = bat_simple["simple_runs"] / bat_simple["simple_balls"]
bat_simple = bat_simple.sort_values("simple_score", ascending=False).reset_index(drop=True)

bowl_simple_src = s10.loc[~(is_wkt & s10["dismissal_type"].isin(non_bowler_dismissals)) & s10["simple_pressure"]]
bowl_simple = (
    bowl_simple_src.groupby("bowler_id", as_index=False)
    .agg(
        simple_balls=("is_legal", "size"),
        simple_legal=("is_legal", "sum"),
        simple_runs=("bowler_runs", "sum"),
        simple_wkts=("is_bowler_wicket", "sum"),
    )
    .rename(columns={"bowler_id": "player_id"})
)
bowl_simple = final_bowl[["player_id", "player_name", "final_score"]].merge(bowl_simple, on="player_id", how="inner")
bowl_simple = bowl_simple.loc[bowl_simple["simple_legal"].ge(min_simple_balls)].copy()
bowl_simple["pressure_econ"] = bowl_simple["simple_runs"] * 6.0 / bowl_simple["simple_legal"]
bowl_simple["pressure_wkt_rate"] = bowl_simple["simple_wkts"] / bowl_simple["simple_legal"]

# Combine lower economy and higher wicket rate on equal z-score footing
z_econ = (bowl_simple["pressure_econ"] - bowl_simple["pressure_econ"].mean()) / bowl_simple["pressure_econ"].std(ddof=0)
z_wkt = (bowl_simple["pressure_wkt_rate"] - bowl_simple["pressure_wkt_rate"].mean()) / bowl_simple["pressure_wkt_rate"].std(ddof=0)
bowl_simple["simple_score"] = 0.5 * (-z_econ) + 0.5 * z_wkt
bowl_simple = bowl_simple.sort_values("simple_score", ascending=False).reset_index(drop=True)

bat_simple_overlap = len(set(final_bat.head(10)["player_id"]) & set(bat_simple.head(10)["player_id"]))
bowl_simple_overlap = len(set(final_bowl.head(10)["player_id"]) & set(bowl_simple.head(10)["player_id"]))
bat_simple_rho = spearmanr(bat_simple["final_score"], bat_simple["simple_score"]).statistic
bowl_simple_rho = spearmanr(bowl_simple["final_score"], bowl_simple["simple_score"]).statistic

comp_summary = pd.DataFrame([
    {
        "role": "Batters",
        "players_compared": len(bat_simple),
        "top10_overlap": f"{bat_simple_overlap}/10",
        "spearman_rho": round(float(bat_simple_rho), 4),
    },
    {
        "role": "Bowlers",
        "players_compared": len(bowl_simple),
        "top10_overlap": f"{bowl_simple_overlap}/10",
        "spearman_rho": round(float(bowl_simple_rho), 4),
    },
])
print(f"\nSimple Composite Index vs Main WPA Ranking (min {min_simple_balls} simple-pressure balls within eligible pool):")
print(comp_summary.to_string(index=False))

print("\nSimple Composite Top 10 Batters (runs/ball):", bat_simple.head(10)["player_name"].tolist())
print("Simple Composite Top 10 Bowlers (econ + wkt rate):", bowl_simple.head(10)["player_name"].tolist())


# ------------------------------------------------------------------
# 3c. Match-to-match consistency and 95% match-level bootstrap CI
# ------------------------------------------------------------------
def match_consistency_and_ci(sub_df, top10_df, id_col, wpa_col, meta, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    prior_mean = meta["prior_mean"]
    k = meta["k"]

    is_press = sub_df["high_pressure"].astype(bool)
    tmp = pd.DataFrame({
        "player_id": sub_df[id_col].values,
        "match_id": sub_df["match_id"].values,
        "is_press": is_press.astype(int).values,
        "wpa_press": np.where(is_press, sub_df[wpa_col].values, 0.0),
    })

    # Aggregate to match level so bootstrap resamples whole matches rather than correlated consecutive balls
    per_match = (
        tmp.loc[tmp["player_id"].isin(top10_df["player_id"])]
        .groupby(["player_id", "match_id"], as_index=False)
        .agg(m_press_balls=("is_press", "sum"), m_press_wpa=("wpa_press", "sum"))
    )

    out = []
    for _, row in top10_df.iterrows():
        pid = row["player_id"]
        pm = per_match.loc[per_match["player_id"].eq(pid)]
        pm_active = pm.loc[pm["m_press_balls"].gt(0)]

        # Consistency denominator uses matches where the player actually faced/bowled a high-pressure ball
        pct_pos = 100.0 * pm_active["m_press_wpa"].gt(0).mean() if len(pm_active) > 0 else np.nan

        # Resample across all matches played to capture both match-to-match performance and pressure volume variance
        balls_arr = pm["m_press_balls"].to_numpy()
        wpa_arr = pm["m_press_wpa"].to_numpy()
        n_m = len(balls_arr)

        idx = rng.integers(0, n_m, size=(n_boot, n_m))
        boot_balls = balls_arr[idx].sum(axis=1)
        boot_wpa = wpa_arr[idx].sum(axis=1)
        boot_shrunk = (boot_wpa + k * prior_mean) / (boot_balls + k)
        ci_lo, ci_hi = np.percentile(boot_shrunk, [2.5, 97.5])

        out.append({
            "rank": int(row["rank"]),
            "player_id": pid,
            "player_name": row["player_name"],
            "matches": int(row["matches"]),
            "press_matches": int(len(pm_active)),
            "pct_pos_press_matches": round(float(pct_pos), 1),
            "shrunk_pressure_wpa": round(float(row["shrunk_pressure_wpa"]), 5),
            "ci95_low": round(float(ci_lo), 5),
            "ci95_high": round(float(ci_hi), 5),
        })
    return pd.DataFrame(out)


bat_ci = match_consistency_and_ci(bat_balls, final_bat.head(10), "striker_id", "batter_wpa", bat_meta)
bowl_ci = match_consistency_and_ci(bowl_balls, final_bowl.head(10), "bowler_id", "bowler_wpa", bowl_meta)

print("\nMatch-to-match consistency and 95% bootstrap CI (1,000 resamples) — Final Top 10 Batters:")
print(bat_ci.to_string(index=False))

print("\nMatch-to-match consistency and 95% bootstrap CI (1,000 resamples) — Final Top 10 Bowlers:")
print(bowl_ci.to_string(index=False))
# ------------------------------------------------------------------
# 4. Export dynamically ID-mapped ball-level table for Stage 11
# ------------------------------------------------------------------
# Apply dynamic id_merge_map on a clean copy of df_wpa so Stage 11 never hard-codes merges
player_id_cols = [
    "striker_id",
    "non_striker_id",
    "batter_id",
    "bowler_id",
    "dismissed_id",
    "dismissed_player_id",
]
df_wpa_mapped = df_wpa.copy()
if id_merge_map:
    for id_col in player_id_cols:
        if id_col in df_wpa_mapped.columns:
            df_wpa_mapped[id_col] = df_wpa_mapped[id_col].replace(id_merge_map)

print(
    f"\nBuilt df_wpa_mapped for Stage 11: shape={df_wpa_mapped.shape} | "
    f"knock_out present={'knock_out' in df_wpa_mapped.columns} | "
    f"norm_wpa present={'norm_wpa' in df_wpa_mapped.columns} | "
    f"dynamic merges applied={len(id_merge_map)}"
)

Duplicate player-name audit:
 player_id     player_name                                teams_in_p  bat_balls_s8  bowl_balls_s8  total_balls_s8 bat_style            bowl_style                         action
      7446     Abdul Samad Lucknow Super Giants, Sunrisers Hyderabad           441              9             450       RHB   Right-arm Leg-Break                        primary
      6922     Abdul Samad                      Pakistan, Pakistan A            63              0              63       RHB                       kept separate (no shared team)
     29833    Bashir Ahmad                               Afghanistan             6             20              26       LHB         Left-arm Fast                        primary
     39479    Bashir Ahmad                             Afghanistan A             0              0               0                                 kept separate (no shared team)
      8638    Basil Hameed                      United Arab Emirates             0    

## Stage 11: Composite Pressure Rankings, Full-Pipeline Match Bootstrap, Sensitivity & Figures

In [22]:
# Stage 11: Composite pressure ranking, full-pipeline match bootstrap, sensitivity checks, knockout diagnostic, and report figures
# Inputs: df_wpa_mapped, id_to_name (from Stage 10), p
# Outputs: final_bat, final_bowl, bat_boot, bowl_boot, weight_sens_df, knockout diagnostics, and fig1-fig4 PNGs

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.metrics import log_loss
from google.colab import files

# Consume the dynamically ID-mapped table from Stage 10 without hard-coding id_merge_map
if "df_wpa_mapped" not in globals():
    raise RuntimeError("Run Stage 10 first to produce df_wpa_mapped and id_to_name.")

df_final = df_wpa_mapped.copy()

# Reuse the dynamically merged player_id -> player_name lookup from Stage 10
if "id_to_name" not in globals():
    merge_map = globals().get("id_merge_map", {})
    if "p" in globals():
        id_to_name = (
            p[["player_id", "player_name"]]
            .dropna(subset=["player_id"])
            .assign(player_id=lambda df: df["player_id"].replace(merge_map))
            .drop_duplicates(subset=["player_id"])
            .set_index("player_id")["player_name"]
        )
    else:
        id_to_name = pd.Series(dtype="object")

# Wides are not balls faced by the batter
bat_balls = df_final.loc[df_final["batter_faced"]].copy()

# Non-bowler dismissals should not credit or penalize the bowler
non_bowler_dismissals = {
    "RunOut",
    "Retire Out",
    "Obstructing the field",
    "Obstracting the field",
}
is_wkt = df_final["is_wicket"].fillna(False).astype(bool)
bowl_balls = df_final.loc[~(is_wkt & df_final["dismissal_type"].isin(non_bowler_dismissals))].copy()


def _zscore(arr):
    arr = np.asarray(arr, dtype=float)
    std = arr.std(ddof=0)
    if std == 0.0 or np.isnan(std):
        return np.zeros_like(arr)
    return (arr - arr.mean()) / std


def build_match_player_panel(sub_df, id_col, fallback_name_col, raw_wpa_col, norm_wpa_col, pressure_col="high_pressure", min_cons_press_balls=3):
    """Aggregate deliveries to (player_id, match_id) once so ranking and full-pipeline bootstrap stay exact and fast."""
    is_press = sub_df[pressure_col].astype(bool).to_numpy()
    raw_v = sub_df[raw_wpa_col].to_numpy(dtype=float)
    norm_v = sub_df[norm_wpa_col].to_numpy(dtype=float)

    tmp = pd.DataFrame({
        "player_id": sub_df[id_col].to_numpy(),
        "match_id": sub_df["match_id"].to_numpy(),
        "n_press": is_press.astype(np.int64),
        "n_ord": (~is_press).astype(np.int64),
        "sum_raw_press": np.where(is_press, raw_v, 0.0),
        "sum_raw_ord": np.where(~is_press, raw_v, 0.0),
        "sum_norm_press": np.where(is_press, norm_v, 0.0),
        "sum_norm_ord": np.where(~is_press, norm_v, 0.0),
    })

    pm = tmp.groupby(["player_id", "match_id"], as_index=False).agg(
        n_total=("n_press", "size"),
        n_press=("n_press", "sum"),
        n_ord=("n_ord", "sum"),
        sum_raw_press=("sum_raw_press", "sum"),
        sum_raw_ord=("sum_raw_ord", "sum"),
        sum_norm_press=("sum_norm_press", "sum"),
        sum_norm_ord=("sum_norm_ord", "sum"),
    )

    # Require >= 3 pressure balls in a match so 1-ball cameos do not distort match consistency
    pm["cons_elig_match"] = pm["n_press"].ge(min_cons_press_balls).astype(np.int64)
    pm["cons_pos_match"] = (pm["cons_elig_match"].eq(1) & pm["sum_norm_press"].gt(0.0)).astype(np.int64)

    fallback_names = (
        sub_df[[id_col, fallback_name_col]]
        .drop_duplicates(subset=[id_col])
        .set_index(id_col)[fallback_name_col]
    )
    return pm, fallback_names


def rank_composite(
    sub_df,
    id_col,
    fallback_name_col,
    raw_wpa_col,
    norm_wpa_col,
    pressure_col="high_pressure",
    min_matches=20,
    min_press=200,
    min_total=400,
    min_cons_press_balls=3,
    weights=(0.60, 0.25, 0.15),
):
    w_press, w_uplift, w_cons = weights
    pm, fallback_names = build_match_player_panel(
        sub_df, id_col, fallback_name_col, raw_wpa_col, norm_wpa_col,
        pressure_col=pressure_col, min_cons_press_balls=min_cons_press_balls,
    )

    agg = pm.groupby("player_id", as_index=False).agg(
        matches=("match_id", "nunique"),
        n_total=("n_total", "sum"),
        n_press=("n_press", "sum"),
        n_ord=("n_ord", "sum"),
        sum_raw_press=("sum_raw_press", "sum"),
        sum_raw_ord=("sum_raw_ord", "sum"),
        sum_norm_press=("sum_norm_press", "sum"),
        sum_norm_ord=("sum_norm_ord", "sum"),
        n_cons_matches=("cons_elig_match", "sum"),
        n_pos_matches=("cons_pos_match", "sum"),
    )

    agg["mean_wpa_pressure"] = np.where(agg["n_press"].gt(0), agg["sum_raw_press"] / agg["n_press"], np.nan)
    agg["mean_wpa_ordinary"] = np.where(agg["n_ord"].gt(0), agg["sum_raw_ord"] / agg["n_ord"], np.nan)
    agg["mean_norm_wpa_pressure"] = np.where(agg["n_press"].gt(0), agg["sum_norm_press"] / agg["n_press"], np.nan)
    agg["mean_norm_wpa_ordinary"] = np.where(agg["n_ord"].gt(0), agg["sum_norm_ord"] / agg["n_ord"], np.nan)
    agg["raw_consistency"] = np.where(agg["n_cons_matches"].gt(0), agg["n_pos_matches"] / agg["n_cons_matches"], np.nan)

    agg["player_name"] = agg["player_id"].map(id_to_name).fillna(agg["player_id"].map(fallback_names))

    # Eligibility filter removes small-sample players prior to empirical Bayes shrinkage
    elig = agg.loc[
        agg["matches"].ge(min_matches)
        & agg["n_press"].ge(min_press)
        & agg["n_total"].ge(min_total)
    ].copy()

    # 1. Shrunk Pressure Normalized WPA
    prior_press = elig["mean_norm_wpa_pressure"].mean()
    k_press = float(elig["n_press"].median())
    elig["shrunk_press_norm_wpa"] = (
        elig["n_press"] * elig["mean_norm_wpa_pressure"] + k_press * prior_press
    ) / (elig["n_press"] + k_press)

    # 2. Floored Uplift (shrink ordinary norm_wpa toward eligible ordinary mean; floor uplift at 0)
    prior_ord = elig["mean_norm_wpa_ordinary"].mean()
    k_ord = float(elig["n_ord"].median())
    elig["shrunk_ord_norm_wpa"] = (
        elig["n_ord"] * elig["mean_norm_wpa_ordinary"] + k_ord * prior_ord
    ) / (elig["n_ord"] + k_ord)
    elig["raw_uplift"] = elig["shrunk_press_norm_wpa"] - elig["shrunk_ord_norm_wpa"]
    elig["floored_uplift"] = np.maximum(0.0, elig["raw_uplift"])

    # 3. Pool-Wide Consistency shrunk toward the eligible pool's actual overall positive-match base rate
    total_cons_m = elig["n_cons_matches"].sum()
    base_rate = float(elig["n_pos_matches"].sum() / total_cons_m) if total_cons_m > 0 else 0.5
    k_cons = float(elig["n_cons_matches"].median())
    elig["shrunk_consistency"] = (
        elig["n_pos_matches"] + k_cons * base_rate
    ) / (elig["n_cons_matches"] + k_cons)

    # Standardize all three components across the eligible pool and form weighted composite score
    elig["z_press"] = _zscore(elig["shrunk_press_norm_wpa"])
    elig["z_uplift"] = _zscore(elig["floored_uplift"])
    elig["z_consistency"] = _zscore(elig["shrunk_consistency"])
    elig["composite_score"] = (
        w_press * elig["z_press"]
        + w_uplift * elig["z_uplift"]
        + w_cons * elig["z_consistency"]
    )

    elig = elig.sort_values("composite_score", ascending=False).reset_index(drop=True)
    elig.insert(0, "rank", np.arange(1, len(elig) + 1))

    meta = {
        "n_eligible": len(elig),
        "prior_press": prior_press,
        "k_press": k_press,
        "prior_ord": prior_ord,
        "k_ord": k_ord,
        "base_rate": base_rate,
        "k_cons": k_cons,
        "weights": weights,
    }
    return elig, meta, pm


final_bat, bat_meta, bat_pm = rank_composite(
    bat_balls, "striker_id", "striker_name", "batter_wpa", "batter_norm_wpa", "high_pressure"
)
final_bowl, bowl_meta, bowl_pm = rank_composite(
    bowl_balls, "bowler_id", "bowler_name", "bowler_wpa", "bowler_norm_wpa", "high_pressure"
)

display_cols = [
    "rank",
    "player_id",
    "player_name",
    "matches",
    "n_press",
    "shrunk_press_norm_wpa",
    "floored_uplift",
    "shrunk_consistency",
    "composite_score",
]

print(
    f"Eligible Batters: {bat_meta['n_eligible']} | prior_press={bat_meta['prior_press']:+.6f} (k={bat_meta['k_press']:.1f}) "
    f"| prior_ord={bat_meta['prior_ord']:+.6f} (k={bat_meta['k_ord']:.1f}) "
    f"| cons_base_rate={100*bat_meta['base_rate']:.2f}% (k={bat_meta['k_cons']:.1f})"
)
print("Final Top 10 Batters (Composite 0.60 / 0.25 / 0.15):")
print(
    final_bat[display_cols].head(10).round({
        "shrunk_press_norm_wpa": 5,
        "floored_uplift": 5,
        "shrunk_consistency": 4,
        "composite_score": 3,
    }).to_string(index=False)
)

print(
    f"\nEligible Bowlers: {bowl_meta['n_eligible']} | prior_press={bowl_meta['prior_press']:+.6f} (k={bowl_meta['k_press']:.1f}) "
    f"| prior_ord={bowl_meta['prior_ord']:+.6f} (k={bowl_meta['k_ord']:.1f}) "
    f"| cons_base_rate={100*bowl_meta['base_rate']:.2f}% (k={bowl_meta['k_cons']:.1f})"
)
print("Final Top 10 Bowlers (Composite 0.60 / 0.25 / 0.15):")
print(
    final_bowl[display_cols].head(10).round({
        "shrunk_press_norm_wpa": 5,
        "floored_uplift": 5,
        "shrunk_consistency": 4,
        "composite_score": 3,
    }).to_string(index=False)
)


# ------------------------------------------------------------------
# Full-Pipeline Match Cluster Bootstrap (n_boot = 1000)
# ------------------------------------------------------------------
def full_pipeline_match_bootstrap(
    pm,
    main_df,
    all_match_ids,
    min_matches=20,
    min_press=200,
    min_total=400,
    weights=(0.60, 0.25, 0.15),
    n_boot=1000,
    seed=42,
):
    """Resample match_ids with replacement and re-run eligibility, shrinkage, z-scoring, and ranking inside each draw."""
    rng = np.random.default_rng(seed)
    w_press, w_uplift, w_cons = weights

    all_pids = np.sort(pm["player_id"].unique())
    n_p = len(all_pids)
    n_m = len(all_match_ids)

    pid_idx = pd.Index(all_pids).get_indexer(pm["player_id"])
    mid_idx = pd.Index(all_match_ids).get_indexer(pm["match_id"])

    # Dense (n_players x n_matches) matrices allow exact full-pipeline resamples via fast matrix-vector products
    M_played = np.zeros((n_p, n_m), dtype=np.float64)
    M_total = np.zeros((n_p, n_m), dtype=np.float64)
    M_press = np.zeros((n_p, n_m), dtype=np.float64)
    M_ord = np.zeros((n_p, n_m), dtype=np.float64)
    M_sum_press = np.zeros((n_p, n_m), dtype=np.float64)
    M_sum_ord = np.zeros((n_p, n_m), dtype=np.float64)
    M_cons_m = np.zeros((n_p, n_m), dtype=np.float64)
    M_pos_m = np.zeros((n_p, n_m), dtype=np.float64)

    M_played[pid_idx, mid_idx] = 1.0
    M_total[pid_idx, mid_idx] = pm["n_total"].to_numpy(dtype=np.float64)
    M_press[pid_idx, mid_idx] = pm["n_press"].to_numpy(dtype=np.float64)
    M_ord[pid_idx, mid_idx] = pm["n_ord"].to_numpy(dtype=np.float64)
    M_sum_press[pid_idx, mid_idx] = pm["sum_norm_press"].to_numpy(dtype=np.float64)
    M_sum_ord[pid_idx, mid_idx] = pm["sum_norm_ord"].to_numpy(dtype=np.float64)
    M_cons_m[pid_idx, mid_idx] = pm["cons_elig_match"].to_numpy(dtype=np.float64)
    M_pos_m[pid_idx, mid_idx] = pm["cons_pos_match"].to_numpy(dtype=np.float64)

    boot_scores = np.full((n_boot, n_p), np.nan, dtype=np.float64)
    boot_ranks = np.full((n_boot, n_p), np.nan, dtype=np.float64)
    probs = np.full(n_m, 1.0 / n_m, dtype=np.float64)

    for b in range(n_boot):
        # Multinomial draw is identical to sampling n_m match_ids with replacement
        w = rng.multinomial(n_m, probs).astype(np.float64)

        b_matches = M_played @ w
        b_total = M_total @ w
        b_press = M_press @ w

        elig_mask = (b_matches >= min_matches) & (b_press >= min_press) & (b_total >= min_total)
        if elig_mask.sum() < 2:
            continue

        e_press = b_press[elig_mask]
        e_ord = (M_ord[elig_mask] @ w)
        e_sum_press = (M_sum_press[elig_mask] @ w)
        e_sum_ord = (M_sum_ord[elig_mask] @ w)
        e_cons_m = (M_cons_m[elig_mask] @ w)
        e_pos_m = (M_pos_m[elig_mask] @ w)

        # Recompute shrinkage priors and k values inside each resample
        mean_p = e_sum_press / e_press
        prior_p = mean_p.mean()
        k_p = np.median(e_press)
        shrunk_p = (e_press * mean_p + k_p * prior_p) / (e_press + k_p)

        has_ord = e_ord > 0
        mean_o = np.where(has_ord, e_sum_ord / np.where(has_ord, e_ord, 1.0), np.nan)
        prior_o = np.nanmean(mean_o)
        mean_o = np.where(has_ord, mean_o, prior_o)
        k_o = np.median(e_ord)
        shrunk_o = (e_ord * mean_o + k_o * prior_o) / (e_ord + k_o)

        uplift = np.maximum(0.0, shrunk_p - shrunk_o)

        tot_cm = e_cons_m.sum()
        base_r = (e_pos_m.sum() / tot_cm) if tot_cm > 0 else 0.5
        k_c = np.median(e_cons_m)
        shrunk_c = (e_pos_m + k_c * base_r) / (e_cons_m + k_c)

        score = w_press * _zscore(shrunk_p) + w_uplift * _zscore(uplift) + w_cons * _zscore(shrunk_c)

        # Rank 1 = highest composite score within this bootstrap draw
        order = np.argsort(-score)
        ranks = np.empty_like(score)
        ranks[order] = np.arange(1, len(score) + 1, dtype=np.float64)

        boot_scores[b, elig_mask] = score
        boot_ranks[b, elig_mask] = ranks

    pid_to_col = {pid: i for i, pid in enumerate(all_pids)}
    rows = []
    for _, row in main_df.iterrows():
        col_i = pid_to_col[row["player_id"]]
        sc_arr = boot_scores[:, col_i]
        rk_arr = boot_ranks[:, col_i]
        valid = ~np.isnan(sc_arr)

        if valid.sum() > 0:
            sc_lo, sc_hi = np.percentile(sc_arr[valid], [2.5, 97.5])
            rk_lo, rk_hi = np.percentile(rk_arr[valid], [2.5, 97.5])
        else:
            sc_lo, sc_hi, rk_lo, rk_hi = np.nan, np.nan, np.nan, np.nan

        # Top-10 probability counts ineligible draws as non-Top-10 so borderline-volume uncertainty is reflected
        pct_top10 = 100.0 * np.mean(np.where(valid, rk_arr <= 10.0, False))
        pct_elig = 100.0 * np.mean(valid)

        rows.append({
            "rank": int(row["rank"]),
            "player_id": row["player_id"],
            "player_name": row["player_name"],
            "matches": int(row["matches"]),
            "n_press": int(row["n_press"]),
            "n_cons_matches": int(row["n_cons_matches"]),
            "raw_consistency_pct": round(100.0 * float(row["raw_consistency"]), 1) if pd.notna(row["raw_consistency"]) else np.nan,
            "shrunk_consistency_pct": round(100.0 * float(row["shrunk_consistency"]), 1),
            "composite_score": round(float(row["composite_score"]), 3),
            "score_ci95_low": round(float(sc_lo), 3),
            "score_ci95_high": round(float(sc_hi), 3),
            "rank_ci95_low": round(float(rk_lo), 1),
            "rank_ci95_high": round(float(rk_hi), 1),
            "pct_top10": round(float(pct_top10), 1),
            "pct_boot_eligible": round(float(pct_elig), 1),
        })
    return pd.DataFrame(rows)


all_match_ids = np.sort(df_final["match_id"].unique())
bat_boot = full_pipeline_match_bootstrap(bat_pm, final_bat, all_match_ids, n_boot=1000, seed=42)
bowl_boot = full_pipeline_match_bootstrap(bowl_pm, final_bowl, all_match_ids, n_boot=1000, seed=42)

boot_print_cols = [
    "rank", "player_id", "player_name", "matches", "n_press",
    "shrunk_consistency_pct", "composite_score",
    "score_ci95_low", "score_ci95_high",
    "rank_ci95_low", "rank_ci95_high", "pct_top10", "pct_boot_eligible",
]
print("\nFull-pipeline cluster bootstrap (1,000 match resamples) — Final Top 10 Batters:")
print(bat_boot[boot_print_cols].head(10).to_string(index=False))

print("\nFull-pipeline cluster bootstrap (1,000 match resamples) — Final Top 10 Bowlers:")
print(bowl_boot[boot_print_cols].head(10).to_string(index=False))


# ------------------------------------------------------------------
# Weight-Sensitivity Check
# ------------------------------------------------------------------
weight_triplets = [
    (0.60, 0.25, 0.15),
    (0.70, 0.20, 0.10),
    (0.50, 0.30, 0.20),
    (1.00, 0.00, 0.00),
]

weight_rows = []
for role, main_df in [("Batters", final_bat), ("Bowlers", final_bowl)]:
    top10_main = set(main_df.head(10)["player_id"])
    for wp_w, wu_w, wc_w in weight_triplets:
        alt_score = (
            wp_w * main_df["z_press"]
            + wu_w * main_df["z_uplift"]
            + wc_w * main_df["z_consistency"]
        )
        alt_top10 = set(main_df.assign(_alt=alt_score).nlargest(10, "_alt")["player_id"])
        overlap = len(top10_main & alt_top10)
        rho = spearmanr(main_df["composite_score"], alt_score).statistic
        weight_rows.append({
            "role": role,
            "weights_(press,uplift,cons)": f"({wp_w:.2f}, {wu_w:.2f}, {wc_w:.2f})",
            "top10_overlap": f"{overlap}/10",
            "spearman_rho": round(float(rho), 4),
        })

weight_sens_df = pd.DataFrame(weight_rows)
print("\nWeight-sensitivity check vs Main (0.60, 0.25, 0.15):")
print(weight_sens_df.to_string(index=False))


# ------------------------------------------------------------------
# Leverage Threshold Sensitivity (Top 20% and Top 35% vs Main 25%)
# ------------------------------------------------------------------
thresh_rows = []
for role, df_role, id_col, name_col, raw_col, norm_col, main_df in [
    ("Batters", bat_balls, "striker_id", "striker_name", "batter_wpa", "batter_norm_wpa", final_bat),
    ("Bowlers", bowl_balls, "bowler_id", "bowler_name", "bowler_wpa", "bowler_norm_wpa", final_bowl),
]:
    top10_main = set(main_df.head(10)["player_id"])
    for col, tag in [("high_pressure_20", "Top 20% leverage"), ("high_pressure_35", "Top 35% leverage")]:
        alt_df, _, _ = rank_composite(df_role, id_col, name_col, raw_col, norm_col, pressure_col=col)
        overlap = len(top10_main & set(alt_df.head(10)["player_id"]))
        merged = main_df[["player_id", "composite_score"]].merge(
            alt_df[["player_id", "composite_score"]],
            on="player_id",
            suffixes=("_main", "_alt"),
        )
        rho = spearmanr(merged["composite_score_main"], merged["composite_score_alt"]).statistic
        thresh_rows.append({
            "role": role,
            "comparison": tag,
            "alt_eligible": len(alt_df),
            "common_eligible": len(merged),
            "top10_overlap": f"{overlap}/10",
            "spearman_rho": round(float(rho), 4),
        })

print("\nLeverage threshold sensitivity vs Main (Top 25% leverage):")
print(pd.DataFrame(thresh_rows).to_string(index=False))


# ------------------------------------------------------------------
# Simple Non-ML Composite Benchmark Comparison
# ------------------------------------------------------------------
simple_press_mask = (
    df_final["innings"].eq(2)
    & (
        (df_final["required_rr"].ge(10.0) & df_final["cum_wickets"].ge(3))
        | (df_final["overid"].ge(16) & df_final["runs_needed"].le(20))
    )
)
min_simple_balls = 30

bat_simple = (
    df_final.loc[df_final["batter_faced"] & simple_press_mask]
    .groupby("striker_id", as_index=False)
    .agg(simple_balls=("batter_runs", "size"), simple_runs=("batter_runs", "sum"))
    .rename(columns={"striker_id": "player_id"})
)
bat_simple = final_bat[["player_id", "player_name", "composite_score"]].merge(bat_simple, on="player_id", how="inner")
bat_simple = bat_simple.loc[bat_simple["simple_balls"].ge(min_simple_balls)].copy()
bat_simple["simple_score"] = bat_simple["simple_runs"] / bat_simple["simple_balls"]
bat_simple = bat_simple.sort_values("simple_score", ascending=False).reset_index(drop=True)

bowl_simple = (
    df_final.loc[~(is_wkt & df_final["dismissal_type"].isin(non_bowler_dismissals)) & simple_press_mask]
    .groupby("bowler_id", as_index=False)
    .agg(
        simple_legal=("is_legal", "sum"),
        simple_runs=("bowler_runs", "sum"),
        simple_wkts=("is_bowler_wicket", "sum"),
    )
    .rename(columns={"bowler_id": "player_id"})
)
bowl_simple = final_bowl[["player_id", "player_name", "composite_score"]].merge(bowl_simple, on="player_id", how="inner")
bowl_simple = bowl_simple.loc[bowl_simple["simple_legal"].ge(min_simple_balls)].copy()
bowl_simple["pressure_econ"] = bowl_simple["simple_runs"] * 6.0 / bowl_simple["simple_legal"]
bowl_simple["pressure_wkt_rate"] = bowl_simple["simple_wkts"] / bowl_simple["simple_legal"]
bowl_simple["simple_score"] = 0.5 * (-_zscore(bowl_simple["pressure_econ"])) + 0.5 * _zscore(bowl_simple["pressure_wkt_rate"])
bowl_simple = bowl_simple.sort_values("simple_score", ascending=False).reset_index(drop=True)

bat_simple_overlap = len(set(final_bat.head(10)["player_id"]) & set(bat_simple.head(10)["player_id"]))
bowl_simple_overlap = len(set(final_bowl.head(10)["player_id"]) & set(bowl_simple.head(10)["player_id"]))
bat_simple_rho = spearmanr(bat_simple["composite_score"], bat_simple["simple_score"]).statistic
bowl_simple_rho = spearmanr(bowl_simple["composite_score"], bowl_simple["simple_score"]).statistic

print(f"\nSimple Non-ML Composite vs Main Composite Ranking (min {min_simple_balls} simple-pressure balls):")
print(pd.DataFrame([
    {"role": "Batters", "players_compared": len(bat_simple), "top10_overlap": f"{bat_simple_overlap}/10", "spearman_rho": round(float(bat_simple_rho), 4)},
    {"role": "Bowlers", "players_compared": len(bowl_simple), "top10_overlap": f"{bowl_simple_overlap}/10", "spearman_rho": round(float(bowl_simple_rho), 4)},
]).to_string(index=False))


# ------------------------------------------------------------------
# Knockout Diagnostic
# ------------------------------------------------------------------
ko_lev = (
    df_final.groupby("knock_out", dropna=False)
    .agg(
        matches=("match_id", "nunique"),
        deliveries=("leverage", "size"),
        mean_leverage=("leverage", "mean"),
        high_pressure_pct=("high_pressure", lambda s: 100.0 * s.mean()),
    )
    .reset_index()
)
print("\nKnockout vs League match leverage diagnostic:")
print(ko_lev.round({"mean_leverage": 4, "high_pressure_pct": 2}).to_string(index=False))

def top10_knockout_breakdown(sub_df, top10_df, id_col, raw_wpa_col, norm_wpa_col):
    is_ko_press = sub_df["knock_out"].fillna(0).astype(int).eq(1) & sub_df["high_pressure"].astype(bool)
    ko_sub = sub_df.loc[is_ko_press & sub_df[id_col].isin(top10_df["player_id"])]
    ko_agg = (
        ko_sub.groupby(id_col, as_index=False)
        .agg(
            ko_matches=("match_id", "nunique"),
            ko_press_balls=(raw_wpa_col, "size"),
            ko_mean_raw_wpa=(raw_wpa_col, "mean"),
            ko_mean_norm_wpa=(norm_wpa_col, "mean"),
        )
        .rename(columns={id_col: "player_id"})
    )
    out = top10_df[["rank", "player_id", "player_name"]].merge(ko_agg, on="player_id", how="left")
    out["ko_matches"] = out["ko_matches"].fillna(0).astype(int)
    out["ko_press_balls"] = out["ko_press_balls"].fillna(0).astype(int)
    return out

bat_ko = top10_knockout_breakdown(bat_balls, final_bat.head(10), "striker_id", "batter_wpa", "batter_norm_wpa")
bowl_ko = top10_knockout_breakdown(bowl_balls, final_bowl.head(10), "bowler_id", "bowler_wpa", "bowler_norm_wpa")

print("\nKnockout high-pressure performance — Final Top 10 Batters:")
print(bat_ko.round({"ko_mean_raw_wpa": 5, "ko_mean_norm_wpa": 5}).to_string(index=False))

print("\nKnockout high-pressure performance — Final Top 10 Bowlers:")
print(bowl_ko.round({"ko_mean_raw_wpa": 5, "ko_mean_norm_wpa": 5}).to_string(index=False))


# ------------------------------------------------------------------
# Export 4 Report PNGs (300 DPI, white background, clean layout)
# ------------------------------------------------------------------
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "axes.labelsize": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

NAVY = "#1F4E79"
TEAL = "#2A9D8F"
CORAL = "#D95F43"
SLATE = "#94A3B8"
DARK = "#1E293B"

# Figure 1: Two-panel calibration curve (Innings 1 vs Innings 2)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), sharex=True, sharey=True)
bins = np.linspace(0.0, 1.0, 11)

cal_specs = [
    (axes[0], 1, "bat_first_won", "Innings 1: Batting-First Win Model", NAVY),
    (axes[1], 2, "chase_won", "Innings 2: Chasing Win Model", TEAL),
]
for ax, inn, target_col, title, color in cal_specs:
    sub = df_final.loc[df_final["innings"].eq(inn)]
    y_true = sub[target_col].to_numpy()
    y_pred = sub["p_bat_win_before"].to_numpy()

    m_ll = log_loss(y_true, y_pred)
    b_ll = log_loss(y_true, np.full(len(y_true), y_true.mean()))

    cal = (
        pd.DataFrame({"pred": y_pred, "actual": y_true})
        .assign(bin=lambda df: pd.cut(df["pred"], bins=bins, include_lowest=True))
        .groupby("bin", observed=False)
        .agg(mean_pred=("pred", "mean"), actual_rate=("actual", "mean"))
        .dropna()
    )
    max_gap = (cal["mean_pred"] - cal["actual_rate"]).abs().max()

    ax.plot([0, 1], [0, 1], linestyle="--", color=SLATE, linewidth=1.3, label="Perfect calibration")
    ax.plot(cal["mean_pred"], cal["actual_rate"], marker="o", color=color, linewidth=2.2, markersize=6.5, label="OOF binned rate")
    ax.set_title(title, pad=10)
    ax.set_xlabel("Predicted Win Probability")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#CBD5E1")

    note = f"OOF Log-Loss: {m_ll:.4f}\nBase Log-Loss: {b_ll:.4f}\nMax Bin Gap: {max_gap:.4f}"
    ax.text(
        0.04, 0.94, note,
        transform=ax.transAxes,
        va="top", ha="left", fontsize=9.5, color=DARK,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#F8FAFC", edgecolor="#CBD5E1"),
    )

axes[0].set_ylabel("Observed Win Rate")
fig.suptitle("Win-Probability Model Calibration (5-Fold Match-Grouped OOF)", fontsize=13.5, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("fig1_calibration.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

# Figure 2: High-pressure vs ordinary validation by phase (smoothed leverage)
phase_order = ["powerplay", "middle", "death"]
phase_labels = ["Powerplay\n(Overs 1-6)", "Middle\n(Overs 7-15)", "Death\n(Overs 16-20)"]

val_agg = (
    df_final.assign(pressure_group=np.where(df_final["high_pressure"], "High-Pressure", "Ordinary"))
    .groupby(["phase", "pressure_group"], observed=True)
    .agg(
        mean_abs_wpa=("abs_wpa", "mean"),
        wicket_pct=("is_team_wicket", lambda s: 100.0 * s.mean()),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
x = np.arange(len(phase_order))
bar_w = 0.35

metric_specs = [
    (axes[0], "mean_abs_wpa", "Mean |WPA| per Delivery", "Mean |WPA| by Phase", "{:.4f}"),
    (axes[1], "wicket_pct", "Wicket Rate (% of Deliveries)", "Wicket Rate (%) by Phase", "{:.2f}%"),
]

for ax, col, ylabel, title, fmt in metric_specs:
    hp_vals = [val_agg.loc[(val_agg["phase"] == ph) & (val_agg["pressure_group"] == "High-Pressure"), col].iloc[0] for ph in phase_order]
    ord_vals = [val_agg.loc[(val_agg["phase"] == ph) & (val_agg["pressure_group"] == "Ordinary"), col].iloc[0] for ph in phase_order]

    b1 = ax.bar(x - bar_w / 2, hp_vals, width=bar_w, color=CORAL, label="High-Pressure (Top 25% Smoothed Leverage)")
    b2 = ax.bar(x + bar_w / 2, ord_vals, width=bar_w, color=SLATE, label="Ordinary (Bottom 75%)")

    max_y = max(hp_vals + ord_vals)
    ax.set_ylim(0, max_y * 1.22)
    for bars in (b1, b2):
        for bar in bars:
            h = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                h + max_y * 0.02,
                fmt.format(h),
                ha="center", va="bottom", fontsize=9, color=DARK, fontweight="bold",
            )

    ax.set_xticks(x)
    ax.set_xticklabels(phase_labels)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=10)
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    ax.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="#CBD5E1", fontsize=8.8)

fig.suptitle("High-Pressure vs. Ordinary Delivery Validation Across Match Phases", fontsize=13.5, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("fig2_pressure_validation.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

# Figure 3: Two-panel horizontal bar chart of Final Top 10 Batters and Bowlers (Composite Score)
fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.6))

rank_specs = [
    (axes[0], final_bat.head(10), "Final Top 10 Batters", NAVY),
    (axes[1], final_bowl.head(10), "Final Top 10 Bowlers", TEAL),
]

for ax, top10, title, color in rank_specs:
    y_pos = np.arange(len(top10))
    vals = top10["composite_score"].to_numpy()
    labels = [f"#{int(r)}  {n}" for r, n in zip(top10["rank"], top10["player_name"])]

    ax.barh(y_pos, vals, height=0.66, color=color, alpha=0.9)
    ax.axvline(0.0, color=DARK, linewidth=0.9, label="Eligible pool mean (0.0)")

    x_min = min(0.0, vals.min()) - 0.15
    x_max = vals.max() * 1.42
    ax.set_xlim(x_min, x_max)

    for i, (val, n_p) in enumerate(zip(vals, top10["n_press"])):
        ax.text(
            max(val, 0.0) + (x_max - x_min) * 0.015,
            i,
            f"{val:+.3f}  (n={int(n_p)})",
            va="center", ha="left", fontsize=9, color=DARK,
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9.8)
    ax.invert_yaxis()
    ax.set_xlabel("Composite Pressure Score (0.60 Press + 0.25 Uplift + 0.15 Consistency)")
    ax.set_title(title, pad=10)
    ax.grid(axis="x", linestyle=":", alpha=0.5)
    ax.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#CBD5E1", fontsize=8.8)

fig.suptitle("Final Composite Pressure Rankings (Min 20 Matches, 200 Pressure Balls, 400 Total Balls)", fontsize=13.5, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("fig3_top10_rankings.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

# Figure 4: Two-panel dot-and-whisker plot with full-pipeline 95% score CIs, rank intervals, and % Top-10
fig, axes = plt.subplots(1, 2, figsize=(15.0, 5.9))

ci_specs = [
    (axes[0], bat_boot.head(10), "Top 10 Batters: Full-Pipeline 95% Score CI", NAVY),
    (axes[1], bowl_boot.head(10), "Top 10 Bowlers: Full-Pipeline 95% Score CI", TEAL),
]

for ax, ci_df, title, color in ci_specs:
    y_pos = np.arange(len(ci_df))
    est = ci_df["composite_score"].to_numpy()
    lo = ci_df["score_ci95_low"].to_numpy()
    hi = ci_df["score_ci95_high"].to_numpy()
    labels = [f"#{int(r)}  {n}" for r, n in zip(ci_df["rank"], ci_df["player_name"])]

    xerr = np.vstack([np.maximum(0.0, est - lo), np.maximum(0.0, hi - est)])
    ax.errorbar(
        est, y_pos, xerr=xerr, fmt="o", color=color, ecolor=color,
        elinewidth=2.0, capsize=4, markersize=6.5, zorder=3,
    )
    ax.axvline(0.0, color=DARK, linestyle="--", linewidth=0.9, label="Eligible pool mean (0.0)")

    span = max(hi.max() - min(lo.min(), 0.0), 1.0)
    x_left = min(lo.min(), 0.0) - span * 0.08
    text_x = hi.max() + span * 0.06
    x_right = text_x + span * 0.52
    ax.set_xlim(x_left, x_right)

    ax.text(text_x, -0.75, "95% Rank | Top-10%", va="bottom", ha="left", fontsize=8.6, fontweight="bold", color=DARK)
    for i, (_, r) in enumerate(ci_df.iterrows()):
        ax.text(
            text_x, i,
            f"[{r['rank_ci95_low']:.0f}, {r['rank_ci95_high']:.0f}] | {r['pct_top10']:.1f}%",
            va="center", ha="left", fontsize=8.8, color=DARK,
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9.8)
    ax.set_ylim(len(ci_df) - 0.3, -1.05)
    ax.set_xlabel("Composite Pressure Score (95% Full-Pipeline Match Bootstrap CI)")
    ax.set_title(title, pad=10)
    ax.grid(axis="x", linestyle=":", alpha=0.5)
    ax.legend(loc="lower left", frameon=True, facecolor="white", edgecolor="#CBD5E1", fontsize=8.5)

fig.suptitle("Reliability of Final Top 10 Rankings (1,000 Full-Pipeline Match-Cluster Resamples)", fontsize=13.5, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig("fig4_reliability_ci.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.close(fig)

png_files = [
    "fig1_calibration.png",
    "fig2_pressure_validation.png",
    "fig3_top10_rankings.png",
    "fig4_reliability_ci.png",
]
print("\nExported PNG figures:")
for fname in png_files:
    print(f"  {fname}: {os.path.getsize(fname) / 1024:.1f} KB")
    files.download(fname)

# ------------------------------------------------------------------
# Uplift Specification Sensitivity (Rank-Percentile & Unfloored Uplift)
# ------------------------------------------------------------------
uplift_rows = []
for role, main_df in [("Batters", final_bat), ("Bowlers", final_bowl)]:
    top10_main = set(main_df.head(10)["player_id"])

    # Alternative 1: rank-percentile normalization for all three components (removes zero-inflation skew)
    pct_score = (
        0.60 * main_df["shrunk_press_norm_wpa"].rank(pct=True)
        + 0.25 * main_df["floored_uplift"].rank(pct=True)
        + 0.15 * main_df["shrunk_consistency"].rank(pct=True)
    )

    # Alternative 2: z-scored unfloored raw_uplift instead of zero-floored uplift
    unfloored_score = (
        0.60 * main_df["z_press"]
        + 0.25 * _zscore(main_df["raw_uplift"])
        + 0.15 * main_df["z_consistency"]
    )

    for label, alt_score in [
        ("Rank-percentile (all 3)", pct_score),
        ("Unfloored raw_uplift (z-score)", unfloored_score),
    ]:
        alt_top10 = set(main_df.assign(_alt=alt_score).nlargest(10, "_alt")["player_id"])
        overlap = len(top10_main & alt_top10)
        rho = spearmanr(main_df["composite_score"], alt_score).statistic
        uplift_rows.append({
            "role": role,
            "specification": label,
            "top10_overlap": f"{overlap}/10",
            "spearman_rho": round(float(rho), 4),
        })

uplift_sens_df = pd.DataFrame(uplift_rows)
print("\nUplift specification sensitivity vs Main (z-scored floored_uplift):")
print(uplift_sens_df.to_string(index=False))

Eligible Batters: 60 | prior_press=+0.000014 (k=316.0) | prior_ord=+0.000364 (k=826.5) | cons_base_rate=54.80% (k=32.5)
Final Top 10 Batters (Composite 0.60 / 0.25 / 0.15):
 rank  player_id          player_name  matches  n_press  shrunk_press_norm_wpa  floored_uplift  shrunk_consistency  composite_score
    1       2367          Travis Head       61      380                0.00176         0.00189              0.5584            2.178
    2       7216    Prabhsimran Singh       56      387                0.00143         0.00168              0.6267            2.020
    3      50284 Vaibhav Sooryavanshi       30      215                0.00210         0.00070              0.5678            1.780
    4        126         Rohit Sharma       86      539                0.00055         0.00195              0.6140            1.514
    5       6281      Abhishek Sharma      122      730                0.00149         0.00039              0.6010            1.274
    6        813          Jos Buttl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  fig2_pressure_validation.png: 240.5 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (3 outputs left out)

  fig4_reliability_ci.png: 441.6 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Uplift specification sensitivity vs Main (z-scored floored_uplift):
   role                  specification top10_overlap  spearman_rho
Batters        Rank-percentile (all 3)          9/10        0.9955
Batters Unfloored raw_uplift (z-score)          9/10        0.9921
Bowlers        Rank-percentile (all 3)          8/10        0.9946
Bowlers Unfloored raw_uplift (z-score)         10/10        0.9929


# Appendix: Archived Validation, Baseline & Diagnostic Checks

## Appendix A (Archived Stage 5): Answer-Key Verification & Public Record Cross-Check

In [23]:
# Stage 5: Validate run/wicket hypotheses against official match results and cross-check player totals
# Inputs: d
# Outputs: Verification summary table, mismatch diagnostics, and Virat Kohli (ID 47) season totals

import re
import numpy as np
import pandas as pd

# Parse decisive match margins from result text (case-insensitive to handle 'Won By' / 'Wickets')
def parse_result(text):
    if pd.isna(text):
        return (None, np.nan)
    m_res = re.search(r"won\s+by\s+(\d+)\s+(runs?|wickets?)", str(text), flags=re.IGNORECASE)
    if not m_res:
        return ("other", np.nan)
    margin = int(m_res.group(1))
    unit = "runs" if m_res.group(2).lower().startswith("run") else "wickets"
    return (unit, margin)

match_meta = (
    d[["match_id", "comp_type", "season", "match_date", "win_team_id", "result"]]
    .drop_duplicates("match_id")
    .copy()
)
parsed = match_meta["result"].apply(parse_result)
match_meta["result_type"] = [x[0] for x in parsed]
match_meta["expected_margin"] = [x[1] for x in parsed]

print("Parsed result_type counts across kept matches:")
print(match_meta["result_type"].value_counts(dropna=False).to_string())

non_standard = match_meta.loc[~match_meta["result_type"].isin(["runs", "wickets"])]
if len(non_standard) > 0:
    print("\nMatches without standard 'Won by N runs/wickets' text:")
    print(non_standard[["match_id", "comp_type", "match_date", "win_team_id", "result"]].to_string(index=False))

# Build alternative run and wicket candidates without mutating Stage 3 columns
runs_s = d["runs"].fillna(0)
ext1_s = d["extras1_runs"].fillna(0)
ext2_s = d["extras_2_runs"].fillna(0)
is_wkt = d["is_wicket"].fillna(False).astype(bool)

run_formulas = {
    "Hyp A (runs+ext1+ext2)": d["team_runs"],
    "Hyp B (runs+ext1)": d["team_runs_hyp_b"],
    "Hyp C (Hyp A excl unlabeled extras)": np.where(d["extras_type"].isna(), runs_s, d["team_runs"]),
}
# Note: test Retire Out included too, since official scorecards count Retire Out as a wicket lost
wkt_formulas = {
    "Wkt A (excl Retire Out)": d["is_team_wicket"],
    "Wkt B (incl Retire Out)": is_wkt,
    "Wkt C (excl RunOut & Retire Out)": is_wkt & ~d["dismissal_type"].isin(["RunOut", "Retire Out"]),
}

# Precompute innings batting team and legal balls per match
inn_base = (
    d.groupby(["match_id", "innings"])
    .agg(
        bat_team=("batting_team_id", "first"),
        legal_balls=("is_legal", "sum"),
    )
    .unstack("innings")
)
inn_base.columns = [f"{col}_inn{inn}" for col, inn in inn_base.columns]
inn_base = inn_base.reset_index()

def evaluate_combo(run_series, wkt_series):
    tmp = pd.DataFrame({
        "match_id": d["match_id"].values,
        "innings": d["innings"].values,
        "tr": run_series,
        "tw": wkt_series.astype(int),
    })
    agg = (
        tmp.groupby(["match_id", "innings"])
        .agg(runs=("tr", "sum"), wkts=("tw", "sum"))
        .unstack("innings")
    )
    agg.columns = [f"{col}_inn{inn}" for col, inn in agg.columns]
    chk = match_meta.merge(inn_base, on="match_id", how="left").merge(agg.reset_index(), on="match_id", how="left")

    decisive = chk.loc[chk["result_type"].isin(["runs", "wickets"])].copy()
    is_run_win = decisive["result_type"].eq("runs")
    is_wkt_win = decisive["result_type"].eq("wickets")

    # Winner by runs check: inn1 - inn2 == N
    run_margin_ok = (decisive["runs_inn1"] - decisive["runs_inn2"]).eq(decisive["expected_margin"])

    # Winner by wickets check: inn2 wickets lost == 10 - N and target reached
    wkt_margin_ok = (
        decisive["wkts_inn2"].eq(10 - decisive["expected_margin"])
        & decisive["runs_inn2"].ge(decisive["runs_inn1"] + 1)
    )

    # Higher-scoring team must match win_team_id
    higher_team = np.where(
        decisive["runs_inn1"].gt(decisive["runs_inn2"]),
        decisive["bat_team_inn1"],
        np.where(decisive["runs_inn2"].gt(decisive["runs_inn1"]), decisive["bat_team_inn2"], np.nan),
    )
    winner_ok = pd.Series(higher_team, index=decisive.index).eq(decisive["win_team_id"])

    decisive["margin_ok"] = np.where(is_run_win, run_margin_ok, wkt_margin_ok)
    decisive["winner_ok"] = winner_ok
    decisive["all_ok"] = decisive["margin_ok"] & decisive["winner_ok"]

    summary = {
        "n_decisive": len(decisive),
        "run_win_pct": 100.0 * run_margin_ok[is_run_win].mean(),
        "wkt_win_pct": 100.0 * wkt_margin_ok[is_wkt_win].mean(),
        "winner_id_pct": 100.0 * winner_ok.mean(),
        "overall_pct": 100.0 * decisive["all_ok"].mean(),
        "mismatches": int((~decisive["all_ok"]).sum()),
    }
    return summary, decisive

rows_summary = []
results_store = {}
for r_name, r_ser in run_formulas.items():
    for w_name, w_ser in wkt_formulas.items():
        s, detail = evaluate_combo(r_ser, w_ser)
        rows_summary.append({"run_formula": r_name, "wkt_formula": w_name, **s})
        results_store[(r_name, w_name)] = detail

summary_df = pd.DataFrame(rows_summary).sort_values("overall_pct", ascending=False)
print("\nAnswer-key verification across run and wicket hypotheses:")
print(
    summary_df.round({"run_win_pct": 2, "wkt_win_pct": 2, "winner_id_pct": 2, "overall_pct": 2})
    .to_string(index=False)
)

# Inspect mismatches under the top-performing formula
best_row = summary_df.iloc[0]
best_key = (best_row["run_formula"], best_row["wkt_formula"])
best_detail = results_store[best_key]
mismatches = best_detail.loc[~best_detail["all_ok"]].copy()

print(f"\nTop formula: {best_key[0]} + {best_key[1]} ({best_row['overall_pct']:.2f}%, {int(best_row['mismatches'])} mismatches)")
print("\nSample 5 mismatched matches under top formula:")
mismatch_cols = [
    "match_id", "comp_type", "match_date", "result", "win_team_id",
    "bat_team_inn1", "bat_team_inn2", "runs_inn1", "runs_inn2",
    "wkts_inn2", "legal_balls_inn1", "legal_balls_inn2", "margin_ok", "winner_ok",
]
print(mismatches[mismatch_cols].head(5).to_string(index=False))

# Check if Hyp A and Hyp B differ on any match outcome
diff_ab = (
    results_store[("Hyp A (runs+ext1+ext2)", best_key[1])]["all_ok"]
    != results_store[("Hyp B (runs+ext1)", best_key[1])]["all_ok"]
)
print(f"\nMatches where Hyp A and Hyp B disagree on pass/fail: {int(diff_ab.sum())}")

# Distribution of legal balls per innings (helps spot unflagged shortened games or missing overs)
inn_balls = d.groupby(["match_id", "innings"])["is_legal"].sum().rename("legal_balls").reset_index()
bins = [-1, 59, 99, 117, 119, 120, 999]
labels = ["<60", "60-99", "100-117", "118-119", "120", ">120"]
inn_balls["ball_bucket"] = pd.cut(inn_balls["legal_balls"], bins=bins, labels=labels)

print("\nLegal balls per innings (bucketed by innings):")
print(pd.crosstab(inn_balls["ball_bucket"], inn_balls["innings"]).to_string())

over_120 = inn_balls.loc[inn_balls["legal_balls"].gt(120)]
print(f"\nInnings with >120 legal balls: {len(over_120)}")
if len(over_120) > 0:
    print(over_120.to_string(index=False))

# Virat Kohli (player_id 47) IPL 2022-2026 cross-check against public career records
vk = d.loc[d["comp_type"].eq("IPL") & d["striker_id"].eq(47)]
vk_by_season = (
    vk.groupby("season")
    .agg(
        matches=("match_id", "nunique"),
        batter_runs=("batter_runs", "sum"),
        balls_faced=("batter_faced", "sum"),
        fours=("is_four", "sum"),
        sixes=("is_six", "sum"),
    )
    .reset_index()
)
vk_by_season["strike_rate"] = (vk_by_season["batter_runs"] * 100.0 / vk_by_season["balls_faced"]).round(2)

print("\nVirat Kohli (player_id 47) — IPL 2022-2026 by season:")
print(vk_by_season.to_string(index=False))
total_vk_runs = int(vk["batter_runs"].sum())
total_vk_balls = int(vk["batter_faced"].sum())
print(
    f"Total IPL 2022-2026: matches={vk['match_id'].nunique()}, "
    f"runs={total_vk_runs}, balls_faced={total_vk_balls}, "
    f"SR={total_vk_runs * 100.0 / total_vk_balls:.2f}"
)

Parsed result_type counts across kept matches:
result_type
runs       386
wickets    382
other       21

Matches without standard 'Won by N runs/wickets' text:
 match_id comp_type match_date  win_team_id                                                            result
    13613      T20I 2022-02-13         20.0                             Australia Won the one over Eliminator
    13614      T20I 2022-02-15         20.0                                           Australia Won by 6 Wkts
    14094       IPL 2022-03-26          8.0                               Kolkata Knight Riders Won by 6 Wkts
    14096       IPL 2022-03-27          4.0                                        Punjab Kings Won by 5 Wkts
    14099       IPL 2022-03-30          5.0                            Royal Challengers Bengaluru Won 3 Wkts
    14100       IPL 2022-03-31       3509.0                                Lucknow Super Giants Won by 6 Wkts
    14101       IPL 2022-04-01          8.0                           

## Appendix B (Archived Stage 6A): Initial Pre-Refinement Snapshot Export

In [24]:
# Appendix B (Archived Stage 6A): Pre-DLS/tie initial snapshot (files.download stripped)
# Inputs: match_flags (Stage 1), d (Stage 3)
# Outputs: exclusion_log_initial.csv, master_deliveries_initial.parquet

import os

exclusion_log_init = match_flags[["match_id", "exclusion_reason"]].copy()
exclusion_log_init_path = "exclusion_log_initial.csv"
exclusion_log_init.to_csv(exclusion_log_init_path, index=False)

master_deliveries_init = d.drop(columns=["team_runs_hyp_b"], errors="ignore").copy()
master_init_path = "master_deliveries_initial.parquet"
try:
    master_deliveries_init.to_parquet(master_init_path, index=False)
except Exception as err:
    print(f"Parquet export failed ({err}); falling back to CSV.")
    master_init_path = "master_deliveries_initial.csv"
    master_deliveries_init.to_csv(master_init_path, index=False)

print(f"Saved {exclusion_log_init_path}: shape={exclusion_log_init.shape}, size={os.path.getsize(exclusion_log_init_path) / 1024:.1f} KB")
print(f"Saved {master_init_path}: shape={master_deliveries_init.shape}, size={os.path.getsize(master_init_path) / (1024 * 1024):.2f} MB")

Saved exclusion_log_initial.csv: shape=(883, 2), size=7.5 KB
Saved master_deliveries_initial.parquet: shape=(188330, 52), size=3.40 MB


## Appendix C (Archived Stage 9): Baseline Empirical Bayes Ratings (Initial Thresholds)

In [25]:
# Stage 9: Compute initial Empirical Bayes shrunk pressure WPA and pressure-lift composite ratings
# Inputs: df_wpa, p
# Outputs: bat_ratings, bowl_ratings (initial eligible player rankings)

import numpy as np
import pandas as pd

# Work on a copy so Stage 8 output stays untouched
s9 = df_wpa.copy()

# Confirm date span matches the 2022-2026 competition window
years = s9["match_date"].dt.year
print(f"Stage 9 input rows: {len(s9):,} | matches: {s9['match_id'].nunique()} | years: {years.min()}-{years.max()}")

# Build a unique player_id -> player_name lookup from p so joins never rely on name strings
id_to_name = (
    p[["player_id", "player_name"]]
    .dropna(subset=["player_id"])
    .drop_duplicates(subset=["player_id"])
    .set_index("player_id")["player_name"]
)

# Batters: wides are not balls faced, so exclude them from batter WPA and ball counts
bat_balls = s9.loc[s9["batter_faced"]].copy()

# Bowlers: exclude non-bowler dismissals (include raw typo 'Obstracting the field')
non_bowler_dismissals = {
    "RunOut",
    "Retire Out",
    "Obstructing the field",
    "Obstracting the field",
}
is_wkt = s9["is_wicket"].fillna(False).astype(bool)
excl_bowl_mask = is_wkt & s9["dismissal_type"].isin(non_bowler_dismissals)

print(f"\nExcluded non-bowler dismissal deliveries: {int(excl_bowl_mask.sum())}")
print(s9.loc[excl_bowl_mask, "dismissal_type"].value_counts().to_string())

bowl_balls = s9.loc[~excl_bowl_mask].copy()


def build_player_ratings(sub_df, id_col, fallback_name_col, wpa_col, role_label):
    is_press = sub_df["high_pressure"].astype(bool)
    tmp = pd.DataFrame({
        "player_id": sub_df[id_col].values,
        "match_id": sub_df["match_id"].values,
        "is_press": is_press.astype(int).values,
        "is_ord": (~is_press).astype(int).values,
        "wpa_press": np.where(is_press, sub_df[wpa_col].values, 0.0),
        "wpa_ord": np.where(~is_press, sub_df[wpa_col].values, 0.0),
    })

    agg = (
        tmp.groupby("player_id", as_index=False)
        .agg(
            matches=("match_id", "nunique"),
            n_total=("is_press", "size"),
            n_press=("is_press", "sum"),
            n_ord=("is_ord", "sum"),
            sum_wpa_press=("wpa_press", "sum"),
            sum_wpa_ord=("wpa_ord", "sum"),
        )
    )

    # Guard division by zero for low-volume players prior to eligibility filtering
    agg["mean_wpa_pressure"] = np.where(agg["n_press"].gt(0), agg["sum_wpa_press"] / agg["n_press"], np.nan)
    agg["mean_wpa_ordinary"] = np.where(agg["n_ord"].gt(0), agg["sum_wpa_ord"] / agg["n_ord"], np.nan)

    # Attach names via player_id; fall back to delivery table name if an ID is absent from p
    fallback_names = (
        sub_df[[id_col, fallback_name_col]]
        .drop_duplicates(subset=[id_col])
        .set_index(id_col)[fallback_name_col]
    )
    agg["player_name"] = agg["player_id"].map(id_to_name).fillna(agg["player_id"].map(fallback_names))

    # Apply minimum volume thresholds so extreme small-sample averages cannot top the rankings
    elig_mask = (
        agg["n_press"].ge(120)
        & agg["n_total"].ge(250)
        & agg["matches"].ge(10)
    )
    elig = agg.loc[elig_mask].copy()

    # Empirical Bayes shrinkage pulls lower-volume eligible players toward the group average
    prior_mean = elig["mean_wpa_pressure"].mean()
    k = elig["n_press"].median()

    elig["shrunk_pressure_wpa"] = (
        elig["n_press"] * elig["mean_wpa_pressure"] + k * prior_mean
    ) / (elig["n_press"] + k)

    # Pressure lift measures whether a player outperforms their own ordinary-situation baseline
    elig["pressure_lift"] = elig["shrunk_pressure_wpa"] - elig["mean_wpa_ordinary"]

    # Standardize across the eligible pool so both components share the same scale
    z_shrunk = (elig["shrunk_pressure_wpa"] - elig["shrunk_pressure_wpa"].mean()) / elig["shrunk_pressure_wpa"].std(ddof=0)
    z_lift = (elig["pressure_lift"] - elig["pressure_lift"].mean()) / elig["pressure_lift"].std(ddof=0)

    elig["z_shrunk_pressure"] = z_shrunk
    elig["z_pressure_lift"] = z_lift
    elig["final_score"] = 0.70 * z_shrunk + 0.30 * z_lift

    elig = elig.sort_values("final_score", ascending=False).reset_index(drop=True)
    elig.insert(0, "rank", np.arange(1, len(elig) + 1))

    print(
        f"\n{role_label}: {len(elig)} eligible out of {len(agg)} total "
        f"| prior_mean={prior_mean:.6f} | k (median n_press)={k:.1f} "
        f"| missing scores={int(elig['final_score'].isna().sum())}"
    )

    display_cols = [
        "rank",
        "player_id",
        "player_name",
        "matches",
        "n_press",
        "mean_wpa_ordinary",
        "mean_wpa_pressure",
        "shrunk_pressure_wpa",
        "final_score",
    ]
    print(f"\nTop 15 {role_label}:")
    print(
        elig[display_cols]
        .head(15)
        .round({
            "mean_wpa_ordinary": 5,
            "mean_wpa_pressure": 5,
            "shrunk_pressure_wpa": 5,
            "final_score": 3,
        })
        .to_string(index=False)
    )
    return elig


bat_ratings = build_player_ratings(bat_balls, "striker_id", "striker_name", "batter_wpa", "Batters")
bowl_ratings = build_player_ratings(bowl_balls, "bowler_id", "bowler_name", "bowler_wpa", "Bowlers")

Stage 9 input rows: 173,920 | matches: 727 | years: 2022-2026

Excluded non-bowler dismissal deliveries: 510
dismissal_type
RunOut                   503
Retire Out                 6
Obstracting the field      1

Batters: 113 eligible out of 645 total | prior_mean=-0.000688 | k (median n_press)=205.0 | missing scores=0

Top 15 Batters:
 rank  player_id          player_name  matches  n_press  mean_wpa_ordinary  mean_wpa_pressure  shrunk_pressure_wpa  final_score
    1       6280          Harry Brook       52      123            0.00061            0.01038              0.00346        2.189
    2        838        Glenn Maxwell       68      134            0.00034            0.00862              0.00299        1.969
    3       6396            Tim David       91      301            0.00270            0.00607              0.00333        1.769
    4      50284 Vaibhav Sooryavanshi       30      215            0.00323            0.00740              0.00345        1.749
    5       7216    Pra

## Appendix D: Individual Player Diagnostic Lookup (Jasprit Bumrah)

In [26]:
# Appendix D: Diagnostic lookup for Jasprit Bumrah (player_id = 2933) from Stage 11 composite and bootstrap tables
# Inputs: final_bowl, bowl_boot, bowl_balls
# Outputs: Printed composite metrics, raw vs normalized WPA, consistency rate, and 95% score/rank intervals

import pandas as pd

TARGET_PID = 2933

if "final_bowl" not in globals() or "bowl_boot" not in globals():
    raise RuntimeError("Run Stage 11 first to populate final_bowl and bowl_boot.")

hit_rank = final_bowl.loc[final_bowl["player_id"].eq(TARGET_PID)]
hit_boot = bowl_boot.loc[bowl_boot["player_id"].eq(TARGET_PID)]

if len(hit_rank) > 0:
    combined = hit_rank.merge(
        hit_boot[
            [
                "player_id",
                "score_ci95_low",
                "score_ci95_high",
                "rank_ci95_low",
                "rank_ci95_high",
                "pct_top10",
                "pct_boot_eligible",
            ]
        ],
        on="player_id",
        how="left",
    )

    print(f"Jasprit Bumrah (player_id={TARGET_PID}) — Stage 11 Composite Summary (out of {len(final_bowl)} eligible bowlers):")
    wpa_cols = [
        "rank",
        "player_id",
        "player_name",
        "matches",
        "n_press",
        "mean_wpa_pressure",
        "mean_norm_wpa_pressure",
        "shrunk_press_norm_wpa",
        "mean_wpa_ordinary",
        "mean_norm_wpa_ordinary",
        "shrunk_ord_norm_wpa",
    ]
    print(
        combined[wpa_cols]
        .round({
            "mean_wpa_pressure": 5,
            "mean_norm_wpa_pressure": 5,
            "shrunk_press_norm_wpa": 5,
            "mean_wpa_ordinary": 5,
            "mean_norm_wpa_ordinary": 5,
            "shrunk_ord_norm_wpa": 5,
        })
        .to_string(index=False)
    )

    comp_cols = [
        "rank",
        "n_cons_matches",
        "raw_consistency",
        "shrunk_consistency",
        "floored_uplift",
        "composite_score",
        "score_ci95_low",
        "score_ci95_high",
        "rank_ci95_low",
        "rank_ci95_high",
        "pct_top10",
        "pct_boot_eligible",
    ]
    print("\nComposite score, consistency, and full-pipeline 95% bootstrap intervals:")
    print(
        combined[comp_cols]
        .round({
            "raw_consistency": 4,
            "shrunk_consistency": 4,
            "floored_uplift": 5,
            "composite_score": 3,
            "score_ci95_low": 3,
            "score_ci95_high": 3,
            "rank_ci95_low": 1,
            "rank_ci95_high": 1,
            "pct_top10": 1,
            "pct_boot_eligible": 1,
        })
        .to_string(index=False)
    )
else:
    print(f"player_id={TARGET_PID} did not meet Stage 11 eligibility thresholds.")
    if "bowl_balls" in globals():
        jb = bowl_balls.loc[bowl_balls["bowler_id"].eq(TARGET_PID)]
        if len(jb) > 0:
            is_p = jb["high_pressure"].astype(bool)
            raw_row = pd.DataFrame([{
                "player_id": TARGET_PID,
                "player_name": jb["bowler_name"].iloc[0],
                "matches": jb["match_id"].nunique(),
                "n_total": len(jb),
                "n_press": int(is_p.sum()),
                "mean_wpa_pressure": jb.loc[is_p, "bowler_wpa"].mean(),
                "mean_norm_wpa_pressure": jb.loc[is_p, "bowler_norm_wpa"].mean(),
                "mean_wpa_ordinary": jb.loc[~is_p, "bowler_wpa"].mean(),
                "mean_norm_wpa_ordinary": jb.loc[~is_p, "bowler_norm_wpa"].mean(),
            }])
            print(raw_row.round(5).to_string(index=False))

Jasprit Bumrah (player_id=2933) — Stage 11 Composite Summary (out of 72 eligible bowlers):
 rank  player_id    player_name  matches  n_press  mean_wpa_pressure  mean_norm_wpa_pressure  shrunk_press_norm_wpa  mean_wpa_ordinary  mean_norm_wpa_ordinary  shrunk_ord_norm_wpa
   58       2933 Jasprit Bumrah       70      523           -0.00057                -0.00082               -0.00043            0.00177                 0.00228              0.00131

Composite score, consistency, and full-pipeline 95% bootstrap intervals:
 rank  n_cons_matches  raw_consistency  shrunk_consistency  floored_uplift  composite_score  score_ci95_low  score_ci95_high  rank_ci95_low  rank_ci95_high  pct_top10  pct_boot_eligible
   58              47           0.4468              0.4512             0.0           -0.759          -1.571            0.335           24.0            72.0        0.1              100.0



### Conclusion and Key Findings

To identify the highest-performing T20 players under pressure without relying on arbitrary scoreline rules, we built a delivery-level Win Probability Added (WPA) framework calibrated via 5-fold match-grouped out-of-fold validation (`OOF Log-Loss = 0.6344` in Innings 1 and `0.3960` in Innings 2). Defining high-pressure states as the top 25% of smoothed delivery leverage and combining Empirical Bayes-shrunk pressure normalized WPA (`60%`), floored pressure uplift (`25%`), and match-level consistency (`15%`) yields a robust, volume-adjusted ranking across 60 eligible batters and 72 eligible bowlers (`>= 20` matches, `>= 200` pressure balls, `>= 400` total balls).

* **ML Leverage Captures True High-Stakes Context:** High-pressure deliveries exhibit substantially larger win-probability swings (`Mean |WPA|` of `0.0176` vs. `0.0124` in the Powerplay, `0.0266` vs. `0.0091` in the Middle overs, and `0.0385` vs. `0.0086` at the Death) and higher dismissal rates (`10.33%` vs. `7.16%` at the Death) than ordinary deliveries. By contrast, a traditional rule-of-thumb pressure filter (`required RR >= 10` with `>= 3` wickets down or `<= 20` runs needed in the final 4 overs) shows weak alignment with the WPA rankings (`2/10` Top-10 overlap; Spearman $\rho = 0.1812$ for batters and $0.2760$ for bowlers), confirming that static heuristics miss critical innings-one and middle-over inflection points.


* **Top Pressure Batters Combine High-Leverage Impact with Uplift and Repeatability:** **Travis Head** (`#1`, composite score `+2.178`, `79.4%` bootstrap Top-10 probability) and **Prabhsimran Singh** (`#2`, `+2.020`, `78.8%` Top-10 probability) lead the batting rankings by pairing strong shrunk pressure WPA (`+0.00176` and `+0.00143`) with substantial positive uplift over their ordinary-ball baselines (`+0.00189` and `+0.00168`). **Vaibhav Sooryavanshi** (`#3`, `+1.780`) posts the highest raw shrunk pressure rate (`+0.00210`), though his smaller sample (`215` pressure balls across `30` matches) places him on the eligibility boundary (`49.8%` unconditional Top-10 probability).


* **Top Pressure Bowlers Are Anchored by High-Impact Pace and Control:** **Lockie Ferguson** (`#1`, `+1.999`), **Harshit Rana** (`#2`, `+1.756`), and **Josh Hazlewood** (`#3`, `+1.750`) top the bowling leaderboard. **Josh Hazlewood** demonstrates the strongest overall statistical reliability among bowlers—backed by the largest high-pressure workload in the Top 10 (`507` pressure balls across `58` matches), a `67.3%` bootstrap Top-10 probability, a strictly positive 95% score confidence interval (`[+0.085, +2.407]`), and positive knockout high-pressure WPA (`+0.00303` across `40` balls).


* **Rankings Are Robust to Weighting and Specification Choices:** Varying component weights between `(0.70, 0.20, 0.10)` and `(0.50, 0.30, 0.20)` preserves `9/10` to `10/10` Top-10 membership with Spearman rank correlations exceeding `0.996`. Similarly, switching to rank-percentile scaling or unfloored uplift retains `8/10` to `10/10` Top-10 overlap (`Spearman ρ > 0.992`), demonstrating that the leaderboard reflects genuine underlying player performance rather than formula sensitivity.
* **Practical Limitations and Scouting Takeaway:** Full-pipeline match-cluster bootstrapping (`1,000` resamples) shows that while the top 2–3 players separate cleanly in Top-10 probability, ranks `#4` through `#10` carry overlapping confidence intervals due to match-to-match T20 variance and small knockout sample sizes (`24` knockout matches total). Decision-makers should therefore pair composite scores with bootstrap Top-10 probabilities and pressure-ball volume when selecting players for high-leverage roles.

Below is a clean, executive-ready **Conclusion & Key Findings** section you can place at the end of your report (right before the Appendix).

---

### Conclusion and Key Findings

To identify the highest-performing T20 players under pressure without relying on arbitrary scoreline rules, we built a delivery-level Win Probability Added (WPA) framework calibrated via 5-fold match-grouped out-of-fold validation (`OOF Log-Loss = 0.6344` in Innings 1 and `0.3960` in Innings 2). Defining high-pressure states as the top 25% of smoothed delivery leverage and combining Empirical Bayes-shrunk pressure normalized WPA (`60%`), floored pressure uplift (`25%`), and match-level consistency (`15%`) yields a robust, volume-adjusted ranking across 60 eligible batters and 72 eligible bowlers (`>= 20` matches, `>= 200` pressure balls, `>= 400` total balls).

* **ML Leverage Captures True High-Stakes Context:** High-pressure deliveries exhibit substantially larger win-probability swings (`Mean |WPA|` of `0.0176` vs. `0.0124` in the Powerplay, `0.0266` vs. `0.0091` in the Middle overs, and `0.0385` vs. `0.0086` at the Death) and higher dismissal rates (`10.33%` vs. `7.16%` at the Death) than ordinary deliveries. By contrast, a traditional rule-of-thumb pressure filter (`required RR >= 10` with `>= 3` wickets down or `<= 20` runs needed in the final 4 overs) shows weak alignment with the WPA rankings (`2/10` Top-10 overlap; Spearman $\rho = 0.1812$ for batters and $0.2760$ for bowlers), confirming that static heuristics miss critical innings-one and middle-over inflection points.


* **Top Pressure Batters Combine High-Leverage Impact with Uplift and Repeatability:** **Travis Head** (`#1`, composite score `+2.178`, `79.4%` bootstrap Top-10 probability) and **Prabhsimran Singh** (`#2`, `+2.020`, `78.8%` Top-10 probability) lead the batting rankings by pairing strong shrunk pressure WPA (`+0.00176` and `+0.00143`) with substantial positive uplift over their ordinary-ball baselines (`+0.00189` and `+0.00168`). **Vaibhav Sooryavanshi** (`#3`, `+1.780`) posts the highest raw shrunk pressure rate (`+0.00210`), though his smaller sample (`215` pressure balls across `30` matches) places him on the eligibility boundary (`49.8%` unconditional Top-10 probability).


* **Top Pressure Bowlers Are Anchored by High-Impact Pace and Control:** **Lockie Ferguson** (`#1`, `+1.999`), **Harshit Rana** (`#2`, `+1.756`), and **Josh Hazlewood** (`#3`, `+1.750`) top the bowling leaderboard. **Josh Hazlewood** demonstrates the strongest overall statistical reliability among bowlers—backed by the largest high-pressure workload in the Top 10 (`507` pressure balls across `58` matches), a `67.3%` bootstrap Top-10 probability, a strictly positive 95% score confidence interval (`[+0.085, +2.407]`), and positive knockout high-pressure WPA (`+0.00303` across `40` balls).


* **Rankings Are Robust to Weighting and Specification Choices:** Varying component weights between `(0.70, 0.20, 0.10)` and `(0.50, 0.30, 0.20)` preserves `9/10` to `10/10` Top-10 membership with Spearman rank correlations exceeding `0.996`. Similarly, switching to rank-percentile scaling or unfloored uplift retains `8/10` to `10/10` Top-10 overlap (`Spearman ρ > 0.992`), demonstrating that the leaderboard reflects genuine underlying player performance rather than formula sensitivity.
* **Practical Limitations and Scouting Takeaway:** Full-pipeline match-cluster bootstrapping (`1,000` resamples) shows that while the top 2–3 players separate cleanly in Top-10 probability, ranks `#4` through `#10` carry overlapping confidence intervals due to match-to-match T20 variance and small knockout sample sizes (`24` knockout matches total). Decision-makers should therefore pair composite scores with bootstrap Top-10 probabilities and pressure-ball volume when selecting players for high-leverage roles.


### Final Top 10 Batters Under Pressure

*Eligibility: $\ge 20$ matches, $\ge 200$ high-pressure balls, $\ge 400$ total balls faced (60 eligible batters)*.

| Rank | Player Name | Matches | Pressure Balls ($n$) | Shrunk Press Norm WPA | Floored Uplift | Shrunk Consistency | Composite Score | 95% Score CI | Top-10 Prob. |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **1** | **Travis Head** | 61 | 380 | +0.00176 | +0.00189 | 55.8% | **+2.178** | [-0.129, +3.012] | 79.4% |
| **2** | **Prabhsimran Singh** | 56 | 387 | +0.00143 | +0.00168 | 62.7% | **+2.020** | [-0.032, +2.782] | 78.8% |
| **3** | **Vaibhav Sooryavanshi** | 30 | 215 | +0.00210 | +0.00070 | 56.8% | **+1.780** | [+0.213, +2.911] | 49.8% |
| **4** | **Rohit Sharma** | 86 | 539 | +0.00055 | +0.00195 | 61.4% | **+1.514** | [-0.346, +2.301] | 64.0% |
| **5** | **Abhishek Sharma** | 122 | 730 | +0.00149 | +0.00039 | 60.1% | **+1.274** | [-0.147, +2.462] | 56.8% |
| **6** | **Jos Buttler** | 115 | 738 | +0.00097 | +0.00064 | 66.5% | **+1.228** | [-0.163, +2.077] | 53.1% |
| **7** | **Yashasvi Jaiswal** | 82 | 623 | +0.00104 | +0.00070 | 60.4% | **+1.135** | [-0.259, +2.095] | 49.6% |
| **8** | **Tim Seifert** | 33 | 253 | +0.00084 | +0.00086 | 57.3% | **+0.993** | [-0.611, +2.361] | 41.3% |
| **9** | **David Warner** | 45 | 284 | +0.00094 | +0.00058 | 60.2% | **+0.992** | [-0.175, +1.869] | 42.6% |
| **10** | **Tim David** | 91 | 301 | +0.00140 | 0.00000 | 58.7% | **+0.955** | [-0.477, +2.627] | 39.8% |

---

### Final Top 10 Bowlers Under Pressure

*Eligibility: $\ge 20$ matches, $\ge 200$ high-pressure balls, $\ge 400$ total deliveries (72 eligible bowlers)*.

| Rank | Player Name | Matches | Pressure Balls ($n$) | Shrunk Press Norm WPA | Floored Uplift | Shrunk Consistency | Composite Score | 95% Score CI | Top-10 Prob. |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| **1** | **Lockie Ferguson** | 51 | 307 | +0.00172 | +0.00233 | 49.6% | **+1.999** | [-0.252, +2.971] | 64.6% |
| **2** | **Harshit Rana** | 41 | 263 | +0.00137 | +0.00216 | 53.1% | **+1.756** | [-0.224, +2.511] | 59.0% |
| **3** | **Josh Hazlewood** | 58 | 507 | +0.00191 | +0.00099 | 55.5% | **+1.750** | [+0.085, +2.407] | 67.3% |
| **4** | **Blessing Muzarabani** | 37 | 319 | +0.00150 | +0.00127 | 52.1% | **+1.455** | [-0.492, +2.738] | 49.0% |
| **5** | **Mujeeb Ur Rahman** | 33 | 221 | +0.00152 | +0.00061 | 59.9% | **+1.424** | [+0.146, +2.211] | 38.1% |
| **6** | **Taskin Ahmed** | 39 | 367 | +0.00135 | +0.00153 | 49.6% | **+1.378** | [-0.287, +2.315] | 48.7% |
| **7** | **Mohammed Shami** | 54 | 395 | +0.00120 | +0.00148 | 54.0% | **+1.367** | [-0.456, +2.482] | 47.0% |
| **8** | **Shaheen Afridi** | 40 | 399 | +0.00113 | +0.00148 | 52.1% | **+1.262** | [-0.479, +2.292] | 40.1% |
| **9** | **Shoriful Islam** | 32 | 251 | +0.00079 | +0.00158 | 51.4% | **+1.018** | [-0.328, +1.925] | 25.4% |
| **10** | **Sunil Narine** | 64 | 248 | +0.00120 | +0.00036 | 53.0% | **+0.868** | [-0.265, +1.978] | 23.7% |

---

### Quick Takeaways for Report Commentary

* **Batters Archetypes:** The top of the batting list splits into two clear profiles: high-uplift top-order aggressors who elevate their game sharply in tense situations (**Travis Head** `+0.00189` uplift, **Prabhsimran Singh** `+0.00168`, **Rohit Sharma** `+0.00195`), and sustained high-impact finishers/openers who post elite raw pressure WPA (**Vaibhav Sooryavanshi** `+0.00210`, **Abhishek Sharma** `+0.00149`, **Tim David** `+0.00140`) or exceptional match-to-match reliability (**Jos Buttler** `66.5%` positive pressure matches).
* **Bowlers Archetypes:** Pace bowlers dominate the Top 10 (`8` of `10` spots), led by high-strike/high-uplift enforcers (**Lockie Ferguson** `+0.00233` uplift, **Harshit Rana** `+0.00216`) alongside high-volume, low-variance control leaders (**Josh Hazlewood** `507` pressure balls, `+0.00191` shrunk pressure WPA, `67.3%` Top-10 probability), complemented by two mystery/finger spinners (**Mujeeb Ur Rahman** `59.9%` consistency and **Sunil Narine**).